# Onde a riva — Definizione dei siti di studio

Un sito alla volta: scegli il punto costiero e quello offshore, carica i profili QGIS, verifica la pendenza della battigia, prepara ERA5 ed esporta un pacchetto per il portale.

**Il notebook è autosufficiente:** le funzioni del progetto sono incluse nella cella tecnica; non occorre clonare GitHub o pubblicare il sito. Parti in modalità dimostrativa per vedere tutti i passaggi senza credenziali. Per i tuoi dati disattiva `DEMO` e usa le celle in ordine.

La pendenza segue **Stockdon_runup_soluzione.ipynb**, celle 24–25: orientamento mare→terra dalla media delle prime/ultime 10 quote; distanza euclidea X/Y dal primo punto a mare; selezione prima del primo punto sopra `Z_MAX`; regressione con `np.polyfit`. Le formule di runup del portale restano invariate.

Il pacchetto finale non contiene token o credenziali. Il notebook non pubblica e non modifica automaticamente la repository.

## 1. Ambiente e cartella di lavoro
Apri in Colab con **File → Carica notebook**. È sufficiente il runtime CPU. L’installazione riguarda soltanto questo runtime.

In [ ]:
#@title Installa le librerie
INSTALLA_DIPENDENZE = True #@param {type:"boolean"}
if INSTALLA_DIPENDENZE:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "numpy>=1.24", "pandas>=2", "matplotlib>=3.7", "folium>=0.14", "pyproj>=3.6", "cdsapi>=0.7.7,<1", "netCDF4>=1.7,<2", "scipy>=1.11,<2", "copernicusmarine==2.4.1", "rasterio>=1.3,<2"])
print("Ambiente pronto.")

In [ ]:
#@title Carica le funzioni del progetto (cella tecnica)
# Codice condiviso con la repository, incluso per funzionare anche prima del push.
from pathlib import Path
import json, sys, tempfile, hashlib
RUNTIME = Path(tempfile.mkdtemp(prefix="ondeariva_runtime_"))
BUNDLED_SOURCES = {'forecast_service/__init__.py': '"""Calcolo e pubblicazione del portale locale, indipendente da Streamlit."""\n', 'forecast_service/site_setup.py': '"""Beach-face slope following Stockdon_runup_soluzione.ipynb, cells 24–25.\n\nThe supplied teaching notebook defines selection and orientation. Diagnostics\nreject degenerate fits; a negative slope is not converted to an absolute value.\n"""\nimport math\n\nBEACHFACE_METHOD_VERSION = \'stockdon_teaching_notebook_v1\'\n\n\ndef prepare_profile(frame, *, distance_col=\'distance\', elevation_col=\'SAMPLE_1\',\n                    x_col=\'X\', y_col=\'Y\', orientation=\'auto\'):\n    import numpy as np\n    import pandas as pd\n    if orientation not in (\'auto\',\'distance_increases_landward\',\'distance_increases_seaward\'):\n        raise ValueError(\'Orientamento non riconosciuto.\')\n    spatial = x_col is not None and y_col is not None\n    if (x_col is None) != (y_col is None):\n        raise ValueError(\'Specificare entrambe le colonne X/Y oppure nessuna.\')\n    required=[distance_col,elevation_col]+([x_col,y_col] if spatial else [])\n    missing=set(required)-set(frame.columns)\n    if missing:\n        raise ValueError(f\'Colonne mancanti: {sorted(missing)}\')\n    profile=frame.copy()\n    for name in required:\n        profile[name]=pd.to_numeric(profile[name],errors=\'raise\')\n    dropped=int(profile[elevation_col].isna().sum())\n    profile=profile.dropna(subset=[elevation_col]).sort_values(distance_col)\n    if len(profile)<2 or not np.isfinite(profile[required].to_numpy(dtype=float)).all():\n        raise ValueError(\'Servono almeno due punti con distanze, coordinate e quote finite.\')\n    if profile[distance_col].duplicated().any():\n        raise ValueError(\'Distanze duplicate nel transetto: verificare il CSV.\')\n    sea_at_end=bool(profile[elevation_col].iloc[:10].mean()>profile[elevation_col].iloc[-10:].mean())\n    ambiguous=bool(abs(profile[elevation_col].iloc[:10].mean()-profile[elevation_col].iloc[-10:].mean())<1e-12)\n    flip=sea_at_end if orientation==\'auto\' else orientation==\'distance_increases_seaward\'\n    if flip:\n        profile=profile.iloc[::-1]\n    profile=profile.reset_index(drop=True)\n    if spatial:\n        profile[\'x_mare\']=np.hypot(profile[x_col]-profile[x_col].iloc[0],profile[y_col]-profile[y_col].iloc[0])\n    else:\n        # Compatibility with 1-D profiles: declared separately in the metadata.\n        profile[\'x_mare\']=(profile[distance_col]-profile[distance_col].iloc[0]).abs()\n    profile.attrs.update(reversed=bool(flip),orientation=orientation,orientation_ambiguous=ambiguous,\n                         dropped_nan_z=dropped,distance_source=\'euclidean_xy\' if spatial else \'one_dimensional_distance\')\n    return profile\n\n\ndef fit_beachface(profile,z_min,z_max,*,elevation_col=\'SAMPLE_1\'):\n    import numpy as np\n    if not math.isfinite(z_min) or not math.isfinite(z_max) or z_min>=z_max:\n        raise ValueError(\'Definire un intervallo di quote crescente.\')\n    # Same sea-connected prefix and regression as pendenza_battigia in cell 25.\n    above=np.where(profile[elevation_col]>z_max)[0]\n    stop=int(above[0]) if len(above) else len(profile)\n    selected=(profile[elevation_col]>=z_min)&(profile.index<stop)\n    x=profile.loc[selected,\'x_mare\'].to_numpy(dtype=float)\n    z=profile.loc[selected,elevation_col].to_numpy(dtype=float)\n    if len(x)<2 or len(np.unique(x))<2:\n        raise ValueError(\'Meno di due distanze distinte prima del primo punto sopra Z_MAX.\')\n    slope,intercept=np.polyfit(x,z,1)\n    if not math.isfinite(slope) or not 0<slope<1:\n        raise ValueError(f\'Pendenza {slope:.6g} non valida: controllare orientamento e tratto di battigia.\')\n    variance=float(np.sum((z-z.mean())**2))\n    r2=1-float(np.sum((z-(slope*x+intercept))**2))/variance if variance>0 else 0.0\n    diagnostics={\'slope\':float(slope),\'n_points\':int(len(x)),\'r_squared\':r2,\'fit_intercept_m\':float(intercept),\n                 \'z_min_m\':float(z_min),\'z_max_m\':float(z_max),\'stop_index\':stop,\n                 \'first_above_zmax_found\':bool(len(above)),\'selected_x_min_m\':float(x.min()),\n                 \'selected_x_max_m\':float(x.max()),\'method\':BEACHFACE_METHOD_VERSION,**profile.attrs}\n    return diagnostics,selected\n\n\ndef analyze_transects(frame,z_min,z_max,*,id_col=\'TR_ID\',distance_col=\'distance\',elevation_col=\'SAMPLE_1\',\n                      x_col=\'X\',y_col=\'Y\',orientation=\'auto\'):\n    import pandas as pd\n    if id_col not in frame or frame[id_col].isna().any():\n        raise ValueError(\'Colonna identificativo assente o con valori mancanti.\')\n    summaries=[]; profiles={}; selections={}\n    for identifier,group in frame.groupby(id_col,sort=False):\n        identifier=str(identifier)\n        try:\n            p=prepare_profile(group,distance_col=distance_col,elevation_col=elevation_col,\n                              x_col=x_col,y_col=y_col,orientation=orientation)\n            summary,selected=fit_beachface(p,z_min,z_max,elevation_col=elevation_col)\n            summaries.append(dict(summary,transect_id=identifier,status=\'valid\',error=\'\'))\n            profiles[identifier]=p; selections[identifier]=selected\n        except (ValueError,KeyError,TypeError) as error:\n            summaries.append({\'transect_id\':identifier,\'status\':\'invalid\',\'error\':str(error),\'slope\':None})\n    return pd.DataFrame(summaries),profiles,selections\n\n\ndef slopes_from_transects(path,z_min,z_max):\n    import pandas as pd\n    frame=pd.read_csv(path)\n    if {\'TR_ID\',\'distance\',\'SAMPLE_1\',\'X\',\'Y\'}<=set(frame):\n        options={}\n    elif {\'transect_id\',\'x_m\',\'z_m\'}<=set(frame):\n        options=dict(id_col=\'transect_id\',distance_col=\'x_m\',elevation_col=\'z_m\',x_col=None,y_col=None)\n    else:\n        raise ValueError(\'CSV QGIS: TR_ID,distance,SAMPLE_1,X,Y; oppure CSV 1-D: transect_id,x_m,z_m.\')\n    summary,_,_=analyze_transects(frame,z_min,z_max,**options)\n    if summary.empty:\n        raise ValueError(\'Il CSV non contiene transetti.\')\n    invalid=summary[summary.status!=\'valid\']\n    if not invalid.empty:\n        raise ValueError(\'; \'.join(f"{r.transect_id}: {r.error}" for r in invalid.itertuples()))\n    return summary.drop(columns=[\'status\',\'error\']).to_dict(\'records\')\n', 'forecast_service/time_utils.py': '"""Pure helpers shared with the standalone Colab setup."""\nimport math\nfrom datetime import datetime,timezone\n\n\ndef iso(dt):\n    return dt.astimezone(timezone.utc).isoformat().replace(\'+00:00\',\'Z\')\n\n\ndef parse_time(value):\n    result=value if isinstance(value,datetime) else datetime.fromisoformat(str(value).replace(\'Z\',\'+00:00\'))\n    return result.replace(tzinfo=timezone.utc) if result.tzinfo is None else result.astimezone(timezone.utc)\n\n\ndef quantile(values,q):\n    ordered=sorted(values)\n    position=(len(ordered)-1)*q\n    low,high=math.floor(position),math.ceil(position)\n    return ordered[low]+(ordered[high]-ordered[low])*(position-low)\n', 'forecast_service/era5_download.py': '"""ERA5 adapter based on wave_climate/download.py, Alessio Rovere (MIT, 2026).\n\nKeep request caching, NetCDF/ZIP handling, nearest sea cell and adaptive chunks.\nOnly waves are needed here. Mean period is never renamed to peak period.\nSee LICENSE-wave-climate.txt for the upstream license.\n"""\nimport datetime as dt\nimport hashlib\nimport io\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport time\nimport zipfile\n\nERA5_TS = \'reanalysis-era5-single-levels-timeseries\'\nVARIABLES = [\'significant_height_of_combined_wind_waves_and_swell\', \'mean_wave_period\', \'mean_wave_direction\']\nSHORT = {\'swh\':\'hs_m\', \'mwp\':\'tm_s\', \'mwd\':\'direction_deg\',\n         \'significant_height_of_combined_wind_waves_and_swell\':\'hs_m\',\n         \'mean_wave_period\':\'tm_s\', \'mean_wave_direction\':\'direction_deg\'}\n\n\nclass TooLarge(Exception):\n    pass\n\n\ndef cds_client():\n    import cdsapi\n    key = os.environ.get(\'CDS_KEY\')\n    if key:\n        return cdsapi.Client(url=os.environ.get(\'CDS_URL\',\'https://cds.climate.copernicus.eu/api\'),key=key,quiet=True)\n    return cdsapi.Client(quiet=True)  # Also supports the user\'s standard ~/.cdsapirc.\n\n\ndef open_cds(path):\n    import xarray as xr\n    if zipfile.is_zipfile(path):\n        frames = []\n        with zipfile.ZipFile(path) as archive:\n            for name in archive.namelist():\n                if name.endswith(\'.nc\'):\n                    with xr.open_dataset(io.BytesIO(archive.read(name))) as ds:\n                        frames.append(ds.load())\n        if not frames:\n            raise ValueError(\'Archivio CDS senza file NetCDF.\')\n        return xr.merge(frames,compat=\'override\')\n    with xr.open_dataset(path) as ds:\n        return ds.load()\n\n\ndef ts_frame(ds):\n    import pandas as pd\n    df=ds.to_dataframe().reset_index()\n    tcol=\'valid_time\' if \'valid_time\' in df else \'time\'\n    keep={v:SHORT[v] for v in ds.data_vars if v in SHORT}\n    if \'hs_m\' not in keep.values():\n        raise ValueError(\'La risposta CDS non contiene Hs.\')\n    out=df[[tcol,*keep,*[c for c in (\'latitude\',\'longitude\') if c in df]]].rename(columns={tcol:\'time\',**keep})\n    out[\'time\']=pd.to_datetime(out[\'time\'],utc=True)\n    return out\n\n\ndef cached_fetch(client, dataset, request, cache, retries=3):\n    cache=Path(cache); cache.mkdir(parents=True,exist_ok=True)\n    key=hashlib.sha256(json.dumps({\'dataset\':dataset,**request},sort_keys=True).encode()).hexdigest()[:16]\n    path=cache/f\'era5_{key}.nc\'\n    if path.exists():\n        return path\n    partial=path.with_suffix(\'.part\')\n    for attempt in range(retries):\n        try:\n            client.retrieve(dataset,request,str(partial))\n            ds=open_cds(partial)\n            try:\n                ts_frame(ds)\n            finally:\n                ds.close()\n            partial.replace(path)\n            return path\n        except Exception as error:\n            partial.unlink(missing_ok=True)\n            message=str(error).lower()\n            if any(word in message for word in (\'too large\',\'cost limit\',\'request too big\')):\n                raise TooLarge() from error\n            if attempt==retries-1:\n                raise\n            time.sleep(30*(attempt+1))\n\n\ndef ts_range(client, lon, lat, start, end, cache, retries=3):\n    request={\'variable\':sorted(VARIABLES),\'location\':{\'longitude\':round(lon,4),\'latitude\':round(lat,4)},\n             \'date\':[f\'{start.isoformat()}/{end.isoformat()}\'],\'data_format\':\'netcdf\'}\n    try:\n        return [cached_fetch(client,ERA5_TS,request,cache,retries)]\n    except TooLarge:\n        if (end-start).days<=31:\n            raise ValueError(\'Richiesta CDS troppo grande anche dopo la suddivisione.\')\n        middle=start+dt.timedelta(days=(end-start).days//2)\n        return ts_range(client,lon,lat,start,middle,cache,retries)+ts_range(client,lon,lat,middle+dt.timedelta(days=1),end,cache,retries)\n\n\ndef nearest_sea_cell(client, lon, lat, cache, search_deg=.75):\n    # CDS time-series accepts a location. Use the gridded product for the area\n    # probe, instead of sending the upstream script\'s area to the point service.\n    request={\'product_type\':[\'reanalysis\'],\'variable\':[VARIABLES[0]],\'year\':[\'2020\'],\'month\':[\'01\'],\n             \'day\':[\'01\'],\'time\':[f\'{h:02d}:00\' for h in range(24)],\n             \'area\':[lat+search_deg,lon-search_deg,lat-search_deg,lon+search_deg],\n             \'data_format\':\'netcdf\',\'download_format\':\'unarchived\'}\n    ds=open_cds(cached_fetch(client,\'reanalysis-era5-single-levels\',request,cache))\n    try:\n        frame=ts_frame(ds)\n    finally:\n        ds.close()\n    cells=frame.groupby([\'latitude\',\'longitude\'])[\'hs_m\'].apply(lambda v:v.notna().mean()).reset_index()\n    cells=cells[cells.hs_m>.9]\n    if cells.empty:\n        raise ValueError(\'Nessuna cella marina ERA5 vicina: scegliere un punto più al largo.\')\n    delta=((cells.longitude-lon+180)%360)-180\n    distance=(delta*math.cos(math.radians(lat)))**2+(cells.latitude-lat)**2\n    cell=cells.loc[distance.idxmin()]\n    return float((cell.longitude+180)%360-180),float(cell.latitude)\n\n\ndef year_chunks(start,end,years=10):\n    if end<start or years<1:\n        raise ValueError(\'Periodo ERA5 non valido.\')\n    chunks=[]\n    current=start\n    while current<=end:\n        stop=min(dt.date(current.year+years-1,12,31),end)\n        chunks.append((current,stop))\n        current=stop+dt.timedelta(days=1)\n    return chunks\n\n\ndef download_series(location, start, end, cache, *, client=None, cell=None):\n    import pandas as pd\n    if end>dt.datetime.now(dt.timezone.utc).date()-dt.timedelta(days=6):\n        raise ValueError(\'Usare una data finale almeno sei giorni prima di oggi per ERA5.\')\n    client=client or cds_client()\n    point=location[\'wave_point\']\n    lon,lat=cell or nearest_sea_cell(client,point[\'longitude\'],point[\'latitude\'],cache)\n    frames=[]\n    for a,b in year_chunks(start,end):\n        print(f\'ERA5 {a} → {b}, cella {lat:.2f} N {lon:.2f} E (cache delle richieste attiva)\')\n        for path in ts_range(client,lon,lat,a,b,cache):\n            ds=open_cds(path)\n            try:\n                frames.append(ts_frame(ds))\n            finally:\n                ds.close()\n    frame=pd.concat(frames).drop_duplicates(\'time\',keep=\'last\').sort_values(\'time\')\n    frame=frame[(frame.time>=pd.Timestamp(start,tz=\'UTC\')) & (frame.time<pd.Timestamp(end+dt.timedelta(days=1),tz=\'UTC\'))]\n    # Reject mixed cells: a local reference must represent one known marine cell.\n    if any(c in frame and frame[c].nunique(dropna=True)>1 for c in (\'latitude\',\'longitude\')):\n        raise ValueError(\'Il CDS ha restituito più celle per la serie puntuale.\')\n    requested_count=(end-start).days*24+24\n    valid=frame[\'hs_m\'].notna()\n    info={\'dataset_id\':ERA5_TS,\'doi\':\'10.24381/1cf1ad76\',\'cell_lon\':lon,\'cell_lat\':lat,\n          \'requested_start\':str(start),\'requested_end\':str(end),\'requested_hours\':requested_count,\n          \'returned_records\':len(frame),\'valid_hs_records\':int(valid.sum()),\n          \'missing_hs_records\':requested_count-int(valid.sum()),\n          \'period_type\':\'mean_wave_period_not_peak\',\'downloaded_at\':dt.datetime.now(dt.timezone.utc).isoformat()}\n    if not valid.any():\n        raise ValueError(\'La cella ERA5 non contiene altezze d’onda valide.\')\n    return frame,info\n', 'forecast_service/climate.py': '"""Offline historical reference, reused by the two daily forecast updates."""\nfrom bisect import bisect_right\nfrom pathlib import Path\nimport csv\nimport hashlib\nimport json\nimport math\n\n\ndef build_climate(csv_path, source, *, time_column=\'time\', hs_column=\'hs_m\'):\n    from forecast_service.time_utils import iso, parse_time, quantile\n    rows = []\n    with Path(csv_path).open(encoding=\'utf-8-sig\', newline=\'\') as handle:\n        for row in csv.DictReader(handle):\n            value = float(row[hs_column])\n            if not math.isfinite(value) or value < 0:\n                raise ValueError(\'La serie storica contiene Hs mancanti o negative: pulire i dati esplicitamente.\')\n            rows.append((parse_time(row[time_column]), value))\n    rows.sort()\n    if len(rows) < 2 or len({t for t,_ in rows}) != len(rows):\n        raise ValueError(\'Servono almeno due tempi distinti, senza duplicati.\')\n    values = sorted(value for _,value in rows)\n    annual = {}\n    for time, value in rows:\n        year = str(time.year)\n        if year not in annual or value > annual[year][\'hs_m\']:\n            annual[year] = {\'hs_m\': value, \'time\': iso(time)}\n    gaps = [(b[0]-a[0]).total_seconds()/3600 for a,b in zip(rows,rows[1:])]\n    step = quantile(gaps, .5)\n    # Irregular sampling would bias an observation-count percentile.\n    if any(abs(g/step-round(g/step)) > .001 for g in gaps) or step <= 0:\n        raise ValueError(\'Campionamento non regolare: uniformare la serie prima del confronto climatologico.\')\n    return {\'schema_version\':1, \'source\':source, \'units\':\'m\', \'start\':iso(rows[0][0]), \'end\':iso(rows[-1][0]),\n            \'n_records\':len(rows), \'time_step_hours\':step,\n            \'coverage_fraction\':len(rows)/(int(round((rows[-1][0]-rows[0][0]).total_seconds()/3600/step))+1),\n            \'hs_sorted_m\':values, \'p95_m\':quantile(values,.95), \'p99_m\':quantile(values,.99),\n            \'max_hs_m\':max(values), \'annual_maxima\':annual,\n            \'input_sha256\':hashlib.sha256(Path(csv_path).read_bytes()).hexdigest(),\n            \'method\':\'Empirical CDF, all seasons and directions; annual maxima are observations, not return levels.\'}\n\n\ndef load_climate(root, relative):\n    path = (root / relative).resolve()\n    if not path.is_relative_to(root.resolve()):\n        raise ValueError(\'Il riferimento climatologico deve essere nella repository.\')\n    data = json.loads(path.read_text())\n    values = data[\'hs_sorted_m\']\n    if data[\'schema_version\'] != 1 or data[\'units\'] != \'m\' or not values or values != sorted(values) or any(not math.isfinite(v) or v < 0 for v in values):\n        raise ValueError(\'Riferimento climatologico non valido.\')\n    if len(values) != data[\'n_records\']:\n        raise ValueError(\'Conteggio della serie storica non coerente.\')\n    from forecast_service.extremes import validate_directional_extremes\n    validate_directional_extremes(data.get(\'directional_extremes\'))\n    return data\n\n\ndef compare_waves(waves, climate):\n    if not climate:\n        return waves, {\'status\':\'not_configured\', \'description\':\'Climatologia non ancora caricata per questa località.\'}\n    values = climate[\'hs_sorted_m\']\n    rows = [dict(wave, hs_percentile=100*bisect_right(values,wave[\'hs_m\'])/len(values)) for wave in waves]\n    summary = {key:value for key,value in climate.items() if key != \'hs_sorted_m\'}\n    summary[\'status\'] = \'available\'\n    summary[\'comparison_note\'] = \'Confronto indicativo fra previsione Copernicus e serie storica; eventuali bias fra prodotti non sono corretti. Il percentile non è un periodo di ritorno.\'\n    from forecast_service.extremes import compare_directional_extremes\n    rows = compare_directional_extremes(rows, climate.get(\'directional_extremes\'))\n    return rows, summary\n', 'forecast_service/exposure.py': '"""User-defined sectors of incoming waves; no nearshore wave transformation."""\nimport math\n\n\ndef is_exposed(direction, exposure):\n    if not exposure or not exposure.get(\'sectors_deg\') or not isinstance(direction, (int, float)) or not math.isfinite(direction):\n        return None\n    direction %= 360\n    return any((a <= direction <= b or b == 360 and direction == 0) if a <= b else (direction >= a or direction <= b)\n               for a, b in exposure[\'sectors_deg\'])\n\n\ndef validate_exposure(exposure):\n    if exposure is None:\n        return\n    if not isinstance(exposure, dict) or not isinstance(exposure.get(\'sectors_deg\'), list) or not exposure[\'sectors_deg\']:\n        raise ValueError(\'Esposizione: indicare almeno un settore oppure lasciare il campo non configurato.\')\n    if not isinstance(exposure.get(\'note\', \'\'), str):\n        raise ValueError(\'La nota sull’esposizione deve essere un testo.\')\n    for pair in exposure[\'sectors_deg\']:\n        if not isinstance(pair, (list, tuple)) or len(pair) != 2:\n            raise ValueError(\'Ogni settore deve contenere due angoli: da, a.\')\n        if any(isinstance(v, bool) or not isinstance(v, (int, float)) or not math.isfinite(v) for v in pair):\n            raise ValueError(\'Gli angoli di esposizione devono essere numeri finiti.\')\n        start, end = pair\n        if not 0 <= start < 360 or not 0 <= end <= 360 or start == end:\n            raise ValueError(\'Settori non validi: inizio da 0 a meno di 360, fine da 0 a 360; estremi distinti. Per tutte le direzioni usare 0,360.\')\n\n\ndef parse_exposure(text, note=\'\'):\n    """Parse clockwise incoming sectors, e.g. \'315,45; 90,180\'."""\n    if not text.strip():\n        return None\n    try:\n        sectors = [[float(value.strip()) for value in sector.split(\',\')] for sector in text.split(\';\')]\n    except ValueError as error:\n        raise ValueError(\'Usare coppie di gradi separate da virgola e settori separati da punto e virgola, es. 315,45; 90,180.\') from error\n    exposure = {\'sectors_deg\': sectors, \'note\': note}\n    validate_exposure(exposure)\n    return exposure\n', 'forecast_service/extremes.py': '"""Directional annual-maxima Gumbel fits, prepared offline during site setup.\n\nDirections are wave provenance, clockwise from true North. Statistical sectors\nform a half-open partition and are distinct from beach exposure sectors.\n"""\nimport calendar\nimport hashlib\nimport math\nfrom pathlib import Path\n\n\ndef sector_index(direction, width_deg):\n    if not isinstance(width_deg, (int, float)) or width_deg not in (30, 45, 60, 90, 120, 180, 360):\n        raise ValueError(\'Ampiezza dei settori ammessa: 30, 45, 60, 90, 120, 180 o 360 gradi.\')\n    if direction is None or not math.isfinite(direction):\n        return None\n    return int(((direction + width_deg/2) % 360)//width_deg)\n\n\ndef gumbel_return_period(height, loc, scale):\n    """Annual-sector exceedance recurrence, not hourly or conditional frequency."""\n    if not all(math.isfinite(v) for v in (height, loc, scale)) or scale <= 0:\n        raise ValueError(\'Parametri Gumbel non validi.\')\n    z=(height-loc)/scale\n    if z < -700:\n        return 1.0\n    if z > 700:\n        return None\n    exceedance=-math.expm1(-math.exp(-z))\n    value=1/exceedance\n    return value if math.isfinite(value) else None\n\n\ndef fit_gumbel(annual_maxima, *, min_years=30, bootstrap_replicates=400, seed=20261105):\n    import numpy as np\n    from scipy.stats import gumbel_r\n    if not isinstance(min_years, int) or min_years < 20:\n        raise ValueError(\'Per questa versione usare almeno 20 anni; il valore predefinito è 30.\')\n    if not isinstance(bootstrap_replicates, int) or bootstrap_replicates < 100:\n        raise ValueError(\'Servono almeno 100 repliche bootstrap.\')\n    values=np.asarray([row[\'hs_m\'] for row in annual_maxima],dtype=float)\n    n=len(values)\n    if len({row[\'year\'] for row in annual_maxima})!=n or not np.isfinite(values).all() or (values<0).any():\n        raise ValueError(\'Massimi annuali non validi o anni duplicati.\')\n    if n < min_years:\n        return {\'status\':\'insufficient_years\',\'n_years\':n,\'reason\':f\'Servono almeno {min_years} anni completi; ne sono disponibili {n}.\'}\n    if np.ptp(values)<=1e-12:\n        return {\'status\':\'constant_maxima\',\'n_years\':n,\'reason\':\'Massimi annuali costanti: non è stimabile una scala Gumbel positiva.\'}\n    loc,scale=map(float,gumbel_r.fit(values))\n    if not math.isfinite(loc) or not math.isfinite(scale) or scale<=0:\n        raise ValueError(\'Il fit Gumbel non produce parametri finiti e scala positiva.\')\n    periods=np.unique(np.r_[np.geomspace(1.01,100,80), [float(n)] if n<=100 else []])\n    central=gumbel_r.isf(1/periods,loc=loc,scale=scale)\n    rng=np.random.default_rng(seed)\n    boot=[]\n    for _ in range(bootstrap_replicates):\n        sample=rng.choice(values,size=n,replace=True)\n        if np.ptp(sample)<=1e-12:\n            continue\n        a,b=map(float,gumbel_r.fit(sample))\n        curve=gumbel_r.isf(1/periods,loc=a,scale=b)\n        if b>0 and np.isfinite(curve).all():\n            boot.append(curve)\n    if len(boot)<.9*bootstrap_replicates:\n        return {\'status\':\'unstable_bootstrap\',\'n_years\':n,\'reason\':\'Fit bootstrap troppo instabile per mostrare una curva con intervalli.\'}\n    low,high=np.quantile(boot,[.025,.975],axis=0)\n    ordered=sorted(annual_maxima,key=lambda row:row[\'hs_m\'],reverse=True)\n    return {\'status\':\'available\',\'distribution\':\'gumbel_r\',\'fit_method\':\'maximum_likelihood\',\n            \'loc_m\':loc,\'scale_m\':scale,\'n_years\':n,\'min_years\':min_years,\n            \'first_year\':min(row[\'year\'] for row in annual_maxima),\'last_year\':max(row[\'year\'] for row in annual_maxima),\n            \'return_period_years\':periods.tolist(),\'return_level_m\':central.tolist(),\n            \'ci95_low_m\':low.tolist(),\'ci95_high_m\':high.tolist(),\n            \'bootstrap_replicates\':len(boot),\'bootstrap_seed\':seed,\n            \'empirical\':[dict(row,return_period_years=(n+1)/rank) for rank,row in enumerate(ordered,1)]}\n\n\ndef build_directional_extremes(csv_path, *, time_column=\'time\', hs_column=\'hs_m\', direction_column=\'direction_deg\',\n                              sector_width_deg=45, min_years=30, bootstrap_replicates=400):\n    import numpy as np\n    import pandas as pd\n    sector_index(0,sector_width_deg)\n    path=Path(csv_path)\n    columns=pd.read_csv(path,nrows=0).columns\n    if direction_column not in columns:\n        return {\'status\':\'not_configured\',\'reason\':f\'Colonna delle direzioni {direction_column!r} assente: il fit settoriale richiede data, Hs e direzione.\'}\n    frame=pd.read_csv(path,usecols=[time_column,hs_column,direction_column]).rename(\n        columns={time_column:\'time\',hs_column:\'hs_m\',direction_column:\'direction_deg\'})\n    frame[\'time\']=pd.to_datetime(frame.time,utc=True,format=\'mixed\')\n    frame=frame.sort_values(\'time\')\n    if frame.empty or frame.time.duplicated().any():\n        raise ValueError(\'La serie degli estremi deve avere tempi distinti e non essere vuota.\')\n    if ((frame.time.dt.minute!=0)|(frame.time.dt.second!=0)|(frame.time.dt.microsecond!=0)).any():\n        return {\'status\':\'unsupported_sampling\',\'reason\':\'Per il fit settoriale usare la serie oraria ERA5 con campioni sulle ore intere.\'}\n    for name in (\'hs_m\',\'direction_deg\'):\n        frame[name]=pd.to_numeric(frame[name],errors=\'coerce\')\n    hs_valid=np.isfinite(frame.hs_m)&(frame.hs_m>=0)\n    direction_valid=np.isfinite(frame.direction_deg)&frame.direction_deg.between(0,360)\n    frame[\'valid\']=hs_valid&((frame.hs_m==0)|direction_valid)\n    frame[\'year\']=frame.time.dt.year\n    width=sector_width_deg\n    frame[\'sector\']=np.floor(((frame.direction_deg+width/2)%360)/width)\n    annual=[]; included=[]\n    for year in range(int(frame.year.min()),int(frame.year.max())+1):\n        group=frame[frame.year==year]\n        expected=8784 if calendar.isleap(year) else 8760\n        valid=int(group.valid.sum())\n        complete=valid==expected\n        annual.append({\'year\':year,\'valid_hours\':valid,\'expected_hours\':expected,\'coverage_fraction\':valid/expected,\'included\':complete})\n        if complete:\n            included.append(year)\n    sectors=[]\n    for index in range(round(360/width)):\n        center=index*width\n        sector={\'id\':f\'sector-{index}\',\'start_deg\':(center-width/2)%360,\'end_deg\':(center+width/2)%360,\n                \'center_deg\':center,\'width_deg\':width,\'annual_maxima\':[]}\n        for year in included:\n            group=frame[(frame.year==year)&frame.valid&(frame.hs_m>0)&(frame.sector==index)]\n            if not group.empty:\n                peak=group.loc[group.hs_m.idxmax()]\n                sector[\'annual_maxima\'].append({\'year\':year,\'hs_m\':float(peak.hs_m),\'time\':peak.time.isoformat().replace(\'+00:00\',\'Z\')})\n        missing=sorted(set(included)-{row[\'year\'] for row in sector[\'annual_maxima\']})\n        if missing:\n            sector[\'fit\']={\'status\':\'years_without_sector_waves\',\'n_years\':len(sector[\'annual_maxima\']),\n                           \'reason\':\'Alcuni anni completi non hanno onde positive da questo settore: escluderli condizionerebbe il tempo di ritorno. Fit non mostrato.\',\n                           \'years_without_waves\':missing}\n        else:\n            sector[\'fit\']=fit_gumbel(sector[\'annual_maxima\'],min_years=min_years,bootstrap_replicates=bootstrap_replicates,seed=20261105+index)\n        sectors.append(sector)\n    result={\'schema_version\':1,\'status\':\'available\',\'direction_convention\':\'from_true_north_clockwise\',\n            \'sector_width_deg\':width,\'sector_boundaries\':\'start_inclusive_end_exclusive\',\n            \'annual_coverage_required\':1.0,\'min_years\':min_years,\'annual_coverage\':annual,\n            \'included_years\':included,\'sectors\':sectors,\'input_sha256\':hashlib.sha256(path.read_bytes()).hexdigest(),\n            \'method\':\'Annual maximum within each directional sector in each complete UTC calendar year; stationary independent annual maxima; Gumbel MLE; percentile bootstrap resampling years.\',\n            \'sampling_note\':\'Only complete hourly calendar years with valid Hs and direction (direction may be missing for calm Hs=0) are used. No-event years are not silently removed.\',\n            \'references\':[\'https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.gumbel_r.html\',\n                          \'https://confluence.ecmwf.int/spaces/FCST/pages/111155338/What+is+the+direction+convention+for+wave+fields\']}\n    validate_directional_extremes(result)\n    return result\n\n\ndef validate_directional_extremes(extremes):\n    if not extremes or extremes.get(\'status\')!=\'available\':\n        return\n    width=extremes[\'sector_width_deg\']\n    sector_index(0,width)\n    if extremes[\'schema_version\']!=1 or extremes[\'direction_convention\']!=\'from_true_north_clockwise\':\n        raise ValueError(\'Formato o convenzione dei settori degli estremi non riconosciuti.\')\n    sectors=extremes[\'sectors\']\n    if len(sectors)!=round(360/width):\n        raise ValueError(\'La partizione dei settori degli estremi non copre tutte le direzioni.\')\n    for index,sector in enumerate(sectors):\n        if sector[\'id\']!=f\'sector-{index}\' or sector[\'center_deg\']!=index*width or sector[\'width_deg\']!=width:\n            raise ValueError(\'Settori degli estremi incoerenti.\')\n        if sector[\'start_deg\']!=(index*width-width/2)%360 or sector[\'end_deg\']!=(index*width+width/2)%360:\n            raise ValueError(\'Estremi dei settori statistici incoerenti.\')\n        fit=sector[\'fit\']\n        if fit[\'status\']!=\'available\':\n            continue\n        if fit[\'n_years\']<20 or fit[\'n_years\']!=len(sector[\'annual_maxima\']):\n            raise ValueError(\'Numero di anni del fit Gumbel non valido.\')\n        if not math.isfinite(fit[\'loc_m\']) or not math.isfinite(fit[\'scale_m\']) or fit[\'scale_m\']<=0:\n            raise ValueError(\'Parametri Gumbel non validi.\')\n        periods=fit[\'return_period_years\']\n        arrays=[fit[key] for key in (\'return_level_m\',\'ci95_low_m\',\'ci95_high_m\')]\n        if not periods or len(set(periods))!=len(periods) or periods!=sorted(periods) or periods[0]<=1:\n            raise ValueError(\'Tempi di ritorno della curva non validi.\')\n        if any(len(a)!=len(periods) or any(not math.isfinite(v) for v in a) for a in arrays):\n            raise ValueError(\'Curve Gumbel non valide.\')\n        if any(a>b for a,b in zip(fit[\'ci95_low_m\'],fit[\'ci95_high_m\'])):\n            raise ValueError(\'Intervalli Gumbel non validi.\')\n\n\ndef compare_directional_extremes(waves, extremes):\n    if not extremes or extremes.get(\'status\')!=\'available\':\n        return waves\n    out=[]\n    for wave in waves:\n        index=sector_index(wave.get(\'direction_deg\'),extremes[\'sector_width_deg\'])\n        sector=extremes[\'sectors\'][index] if index is not None else None\n        fit=sector[\'fit\'] if sector else None\n        period=gumbel_return_period(wave[\'hs_m\'],fit[\'loc_m\'],fit[\'scale_m\']) if fit and fit[\'status\']==\'available\' else None\n        out.append(dict(wave,wave_sector_id=sector[\'id\'] if sector else None,hs_return_period_years=period))\n    return out\n', 'forecast_service/sealevel_reference.py': '"""Fixed historical Copernicus SSH reference, prepared once per site/cell/year."""\nimport calendar\nimport copy\nfrom datetime import datetime, timedelta, timezone\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\n\nDATASET_ID=\'cmems_mod_med_phy-ssh_anfc_4.2km-2D_PT1H-m\'\n\n\ndef _time(value):\n    value=datetime.fromisoformat(str(value).replace(\'Z\',\'+00:00\'))\n    return value.replace(tzinfo=timezone.utc) if value.tzinfo is None else value.astimezone(timezone.utc)\n\n\ndef _iso(value):\n    return value.astimezone(timezone.utc).isoformat().replace(\'+00:00\',\'Z\')\n\n\ndef validate_reference(reference, *, location=None, dataset_id=None, selected_point=None):\n    if reference is None:\n        return\n    if reference.get(\'schema_version\')!=1 or reference.get(\'kind\')!=\'copernicus_historical_mean\':\n        raise ValueError(\'Riferimento storico del livello marino non riconosciuto.\')\n    year=reference[\'year\']\n    if isinstance(year,bool) or not isinstance(year,int) or not 1900<=year<=9998:\n        raise ValueError(\'Anno del riferimento marino non valido.\')\n    expected=8784 if calendar.isleap(year) else 8760\n    start=datetime(year,1,1,tzinfo=timezone.utc)\n    if reference[\'n_records\']!=expected or _time(reference[\'start\'])!=start or _time(reference[\'end\'])!=datetime(year+1,1,1,tzinfo=timezone.utc)-timedelta(hours=1):\n        raise ValueError(\'Il riferimento deve usare un intero anno orario completo.\')\n    if not math.isfinite(reference[\'mean_ssh_m\']) or reference.get(\'variable\')!=\'zos\' or not reference.get(\'dataset_version\'):\n        raise ValueError(\'Valore o provenienza del riferimento SSH non validi.\')\n    if reference.get(\'data_mode\') not in (\'study\',\'demo\'):\n        raise ValueError(\'Modalità del riferimento marino non valida.\')\n    if reference.get(\'local_datum_status\')!=\'not_validated\':\n        raise ValueError(\'La media del modello non certifica il riferimento delle quote locali.\')\n    for field in (\'requested_point\',\'selected_point\'):\n        point=reference[field]\n        if not all(math.isfinite(point[key]) for key in (\'latitude\',\'longitude\')):\n            raise ValueError(\'Coordinate del riferimento marino non valide.\')\n    if dataset_id is not None and reference[\'dataset_id\']!=dataset_id:\n        raise ValueError(\'Riferimento storico e previsione usano dataset SSH diversi: ricalcolare il riferimento.\')\n    if location is not None:\n        point=location.get(\'sealevel_point\',location[\'wave_point\'])\n        if any(abs(point[key]-reference[\'requested_point\'][key])>1e-8 for key in (\'latitude\',\'longitude\')):\n            raise ValueError(\'Il punto del livello marino è cambiato: ricalcolare il riferimento storico per questa località.\')\n        if location.get(\'mean_reference_m\') is not None and abs(location[\'mean_reference_m\']-reference[\'mean_ssh_m\'])>1e-9:\n            raise ValueError(\'Media SSH e riferimento salvato nella località non coincidono.\')\n    if selected_point is not None and any(abs(selected_point[key]-reference[\'selected_point\'][key])>1e-5 for key in (\'latitude\',\'longitude\')):\n        raise ValueError(\'La cella SSH prevista differisce da quella del riferimento storico: ricalcolare il riferimento.\')\n\n\ndef build_reference(records, year, requested_point, selected_point, *, dataset_id=DATASET_ID, dataset_version,\n                    data_mode=\'study\'):\n    expected=8784 if calendar.isleap(year) else 8760\n    samples=sorted((_time(row[\'time\']),float(row[\'ssh_m\'])) for row in records)\n    start=datetime(year,1,1,tzinfo=timezone.utc)\n    if len(samples)!=expected or any(time!=start+timedelta(hours=i) or not math.isfinite(value) for i,(time,value) in enumerate(samples)):\n        raise ValueError(\'SSH storica incompleta: servono tutte le ore dell’anno, senza duplicati o valori mancanti.\')\n    monthly={}\n    for month in range(1,13):\n        values=[value for time,value in samples if time.month==month]\n        monthly[str(month)]={\'mean_ssh_m\':math.fsum(values)/len(values),\'n_records\':len(values)}\n    canonical=json.dumps([[_iso(time),value] for time,value in samples],allow_nan=False,separators=(\',\',\':\'))\n    reference={\'schema_version\':1,\'kind\':\'copernicus_historical_mean\',\'year\':year,\'dataset_id\':dataset_id,\n               \'dataset_version\':dataset_version,\'variable\':\'zos\',\'requested_point\':dict(requested_point),\n               \'selected_point\':dict(selected_point),\'start\':_iso(start),\'end\':_iso(samples[-1][0]),\n               \'n_records\':expected,\'mean_ssh_m\':math.fsum(value for _,value in samples)/expected,\n               \'monthly\':monthly,\'data_mode\':data_mode,\'local_datum_status\':\'not_validated\',\n               \'method\':\'Arithmetic mean of all hourly SSH values in one complete UTC calendar year; fixed for future forecasts.\',\n               \'input_sha256\':hashlib.sha256(canonical.encode()).hexdigest()}\n    validate_reference(reference)\n    return reference\n\n\ndef apply_reference(location, reference):\n    result=copy.deepcopy(location)\n    result.update(mean_reference_m=reference[\'mean_ssh_m\'],sea_level_reference=copy.deepcopy(reference),\n                  vertical_reference_status=\'provisional_model_mean\')\n    validate_location_reference(result)\n    return result\n\n\ndef validate_location_reference(location, *, dataset_id=None, selected_point=None):\n    reference=location.get(\'sea_level_reference\')\n    if reference is None:\n        return\n    if location.get(\'mean_reference_m\') is None:\n        raise ValueError(\'Il riferimento storico richiede la media SSH fissa nella configurazione.\')\n    validate_reference(reference,location=location,dataset_id=dataset_id,selected_point=selected_point)\n\n\ndef download_reference(location, year, username, password, *, dataset_id=DATASET_ID, dataset_version=None, cache=None):\n    """Only called at site setup; no historical download in forecast updates."""\n    import copernicusmarine\n    import numpy as np\n    import pandas as pd\n    if year>=datetime.now(timezone.utc).year:\n        raise ValueError(\'Scegliere un anno di calendario già concluso.\')\n    point=location.get(\'sealevel_point\',location[\'wave_point\'])\n    cache=Path(cache) if cache else None\n    key=hashlib.sha256(json.dumps([dataset_id,dataset_version,year,point],sort_keys=True).encode()).hexdigest()[:16]\n    target=cache/f\'ssh_reference_{year}_{key}.json\' if cache else None\n    if target and target.exists():\n        reference=json.loads(target.read_text())\n        validate_reference(reference,location=location,dataset_id=dataset_id)\n        return reference\n    if not username or not password:\n        raise ValueError(\'Il calcolo storico SSH richiede le credenziali Copernicus Marine.\')\n    if not dataset_version:\n        catalogue=copernicusmarine.describe(dataset_id=dataset_id,disable_progress_bar=True)\n        dataset=next(d for product in catalogue.products for d in product.datasets if d.dataset_id==dataset_id)\n        dataset_version=max(version.label for version in dataset.versions)\n    start=datetime(year,1,1,tzinfo=timezone.utc)\n    ds=copernicusmarine.open_dataset(dataset_id=dataset_id,dataset_version=dataset_version,variables=[\'zos\'],\n        username=username,password=password,start_datetime=_iso(start),\n        end_datetime=_iso(datetime(year+1,1,1,tzinfo=timezone.utc)-timedelta(hours=1)),\n        minimum_latitude=point[\'latitude\'],maximum_latitude=point[\'latitude\'],\n        minimum_longitude=point[\'longitude\'],maximum_longitude=point[\'longitude\'],coordinates_selection_method=\'nearest\')\n    try:\n        array=ds[\'zos\']\n        if array.attrs.get(\'units\',\'m\') not in (\'m\',\'metre\',\'meter\'):\n            raise ValueError(\'La SSH storica deve essere espressa in metri.\')\n        for dim in list(array.dims):\n            if dim!=\'time\':\n                if array.sizes[dim]!=1: raise ValueError(\'Selezionare una sola cella SSH.\')\n                array=array.isel({dim:0})\n        selected={key:float(np.asarray(ds[key].values).reshape(-1)[0]) for key in (\'latitude\',\'longitude\')}\n        times=pd.to_datetime(ds.time.values,utc=True)\n        records=[{\'time\':time.isoformat(),\'ssh_m\':float(value)} for time,value in zip(times,np.asarray(array.values).reshape(-1))]\n    finally:\n        ds.close()\n    reference=build_reference(records,year,point,selected,dataset_id=dataset_id,dataset_version=dataset_version)\n    if target:\n        cache.mkdir(parents=True,exist_ok=True)\n        temporary=target.with_suffix(\'.tmp\');temporary.write_text(json.dumps(reference,ensure_ascii=False,allow_nan=False,indent=2)+\'\\n\');temporary.replace(target)\n    return reference\n', 'forecast_service/site_package.py': '"""Portable study-site packages exported by the standalone Colab notebook."""\nimport copy\nimport json\nfrom pathlib import Path\nimport re\nimport tempfile\nimport zipfile\n\n\ndef validate_payload(payload):\n    slug=payload.get(\'slug\',\'\')\n    if not re.fullmatch(\'[a-z0-9]+(?:-[a-z0-9]+)*\',slug):\n        raise ValueError(\'Identificativo della località non valido.\')\n    if payload.get(\'schema_version\')!=1 or payload.get(\'data_mode\') not in (\'study\',\'demo\'):\n        raise ValueError(\'Formato del pacchetto non riconosciuto.\')\n    if payload.get(\'data_mode\')!=\'demo\' and any(payload.get(\'provenance\',{}).get(key)==\'demo\' for key in (\'profile_data_mode\',\'climate_data_mode\')):\n        raise ValueError(\'Dati dimostrativi dichiarati come dati di studio.\')\n    location=payload[\'location\']\n    from forecast_service.sealevel_reference import validate_location_reference\n    validate_location_reference(location)\n    if payload.get(\'data_mode\')!=\'demo\' and (location.get(\'sea_level_reference\') or {}).get(\'data_mode\')==\'demo\':\n        raise ValueError(\'Riferimento SSH dimostrativo dichiarato come dato di studio.\')\n    from forecast_service.exposure import validate_exposure\n    validate_exposure(location.get(\'wave_exposure\'))\n    if not location.get(\'slopes\') or any(not isinstance(s,(int,float)) or not 0<s<1 for s in location[\'slopes\']):\n        raise ValueError(\'Pendenze non valide.\')\n    if not payload.get(\'thresholds\'):\n        raise ValueError(\'Soglie mancanti.\')\n    if location.get(\'climatology_file\') not in (None,f\'config/climatology/{slug}.json\'):\n        raise ValueError(\'Percorso climatologico non valido nel pacchetto.\')\n    from forecast_service.sectors import resolve_sectors, validate_mappings\n    resolve_sectors(location, payload[\'thresholds\'])\n    validate_mappings(location, slug)\n    json.dumps(payload,allow_nan=False)\n    return slug\n\n\ndef write_site_package(output,payload,*,climatology=None,attachments=None,cartography=None):\n    slug=validate_payload(payload)\n    if bool(climatology) != bool(payload[\'location\'].get(\'climatology_file\')):\n        raise ValueError(\'Riferimento climatologico e configurazione non coerenti.\')\n    if climatology:\n        from forecast_service.climate import load_climate\n        with tempfile.TemporaryDirectory() as folder:\n            root=Path(folder)\n            (root/\'climatology.json\').write_text(json.dumps(climatology,allow_nan=False))\n            load_climate(root,\'climatology.json\')\n    expected=cartography_paths(payload[\'location\'],slug)\n    if set(cartography or {})!=set(expected):\n        raise ValueError(\'Allegati cartografici e configurazione non coerenti.\')\n    validate_cartography(cartography or {})\n    output=Path(output); output.parent.mkdir(parents=True,exist_ok=True)\n    temporary=output.with_suffix(\'.zip.tmp\')\n    try:\n        with zipfile.ZipFile(temporary,\'w\',compression=zipfile.ZIP_DEFLATED) as archive:\n            archive.writestr(\'site.json\',json.dumps(payload,ensure_ascii=False,allow_nan=False,indent=2)+\'\\n\')\n            if climatology:\n                archive.writestr(\'climatology.json\',json.dumps(climatology,ensure_ascii=False,allow_nan=False,separators=(\',\',\':\'))+\'\\n\')\n            for name,path in (cartography or {}).items():\n                archive.write(path,name)\n            for name,path in (attachments or {}).items():\n                if not re.fullmatch(\'[A-Za-z0-9_.-]+\',name) or name.startswith(\'.\'):\n                    raise ValueError(\'Nome allegato non valido.\')\n                archive.write(path,\'diagnostics/\'+name)\n        temporary.replace(output)\n    finally:\n        temporary.unlink(missing_ok=True)\n    return output\n\n\ndef import_site_package(path,root,*,replace=False,allow_demo=False):\n    from forecast_service.pipeline import load_configuration\n    from forecast_service.climate import load_climate\n    root=Path(root)\n    locations,impacts=load_configuration(root)\n    with zipfile.ZipFile(path) as archive:\n        members=archive.infolist()\n        if len(members)>1000 or sum(m.file_size for m in members)>100*1024*1024:\n            raise ValueError(\'Pacchetto troppo grande.\')\n        if len({m.filename for m in members})!=len(members):\n            raise ValueError(\'Nomi duplicati nel pacchetto.\')\n        for member in members:\n            name=member.filename\n            if name not in (\'site.json\',\'climatology.json\') and not re.fullmatch(r\'diagnostics/[A-Za-z0-9_][A-Za-z0-9_.-]*\',name) and not re.fullmatch(r\'cartography/[a-z0-9]+(?:-[a-z0-9]+)*/(?:grid\\.npz|orthophoto\\.png)\',name):\n                raise ValueError(\'Percorso non ammesso nel pacchetto.\')\n        payload=json.loads(archive.read(\'site.json\'))\n        slug=validate_payload(payload)\n        if payload[\'data_mode\']==\'demo\' and not allow_demo:\n            raise ValueError(\'Pacchetto dimostrativo: usare --allow-demo soltanto per una prova locale.\')\n        if slug in locations[\'locations\'] and not replace:\n            raise ValueError(\'Località già presente: usare --replace per aggiornarla intenzionalmente.\')\n        updates={}\n        local=copy.deepcopy(payload[\'location\'])\n        existing=locations[\'locations\'].get(slug,{})\n        if \'sea_level_reference\' not in local and \'mean_reference_m\' not in local and existing.get(\'sea_level_reference\'):\n            from forecast_service.sealevel_reference import apply_reference\n            local=apply_reference(local,existing[\'sea_level_reference\'])\n            payload.setdefault(\'provenance\',{})[\'retained_existing_sea_level_reference\']=True\n        payload[\'location\']=local\n        validate_payload(payload)\n        expected=cartography_paths(local,slug)\n        actual={m.filename for m in members if m.filename.startswith(\'cartography/\')}\n        if actual!=set(expected):\n            raise ValueError(\'Allegati cartografici e configurazione non coerenti.\')\n        with tempfile.TemporaryDirectory() as folder:\n            paths={}\n            for name,destination in expected.items():\n                raw=archive.read(name)\n                target=Path(folder)/name; target.parent.mkdir(parents=True,exist_ok=True); target.write_bytes(raw)\n                paths[name]=target; updates[destination]=raw\n            validate_cartography(paths)\n        if local.get(\'sea_level_reference\'):\n\n            ref=local[\'sea_level_reference\']\n            updates[f"config/study_sites/{slug}/sea_level_reference_{ref[\'year\']}.json"]=(json.dumps(ref,ensure_ascii=False,allow_nan=False,indent=2)+\'\\n\').encode()\n        if local.get(\'climatology_file\'):\n            raw=archive.read(\'climatology.json\')\n            with tempfile.TemporaryDirectory() as folder:\n                climate_root=Path(folder)\n                (climate_root/\'climatology.json\').write_bytes(raw)\n                load_climate(climate_root,\'climatology.json\')\n            updates[local[\'climatology_file\']]=raw\n        elif \'climatology.json\' in archive.namelist():\n            raise ValueError(\'Climatologia nel pacchetto ma non nella configurazione.\')\n        locations[\'locations\'][slug]=local\n        impacts[\'locations\'][slug]={\'name\':local[\'name\'],\'thresholds\':payload[\'thresholds\']}\n        for name,contents in [(\'locations.json\',locations),(\'impact_thresholds.json\',impacts)]:\n            updates[\'config/\'+name]=(json.dumps(contents,ensure_ascii=False,allow_nan=False,indent=2)+\'\\n\').encode()\n        updates[f\'config/study_sites/{slug}/site.json\']=(json.dumps(payload,ensure_ascii=False,allow_nan=False,indent=2)+\'\\n\').encode()\n        for member in members:\n            if member.filename.startswith(\'diagnostics/\'):\n                updates[f\'config/study_sites/{slug}/\'+member.filename]=archive.read(member)\n    # Validate merged configuration before touching any existing file.\n    with tempfile.TemporaryDirectory() as folder:\n        check=Path(folder); (check/\'config\').mkdir()\n        for name in (\'locations.json\',\'impact_thresholds.json\'):\n            (check/\'config\'/name).write_bytes(updates[\'config/\'+name])\n        load_configuration(check)\n    old={name:(root/name).read_bytes() if (root/name).exists() else None for name in updates}\n    try:\n        for name,contents in updates.items():\n            destination=root/name; destination.parent.mkdir(parents=True,exist_ok=True)\n            temporary=destination.with_name(destination.name+\'.import.tmp\')\n            temporary.write_bytes(contents); temporary.replace(destination)\n    except Exception:\n        for name,contents in old.items():\n            destination=root/name\n            if contents is None:\n                destination.unlink(missing_ok=True)\n            else:\n                destination.write_bytes(contents)\n        raise\n    return slug\n\n\ndef cartography_paths(location,slug):\n    from forecast_service.sectors import mapping_entries\n    paths={}\n    for key,mapping in mapping_entries(location):\n        paths[f\'cartography/{key}/grid.npz\']=mapping[\'grid_file\']\n        if mapping.get(\'orthophoto_file\'):\n            paths[f\'cartography/{key}/orthophoto.png\']=mapping[\'orthophoto_file\']\n    return paths\n\n\ndef validate_cartography(paths):\n    from forecast_service.inundation import load_grid\n    import struct\n    for name,path in paths.items():\n        if name.endswith(\'/grid.npz\'):\n            z,_,_,_=load_grid(path)\n            ortho=paths.get(name.replace(\'grid.npz\',\'orthophoto.png\'))\n            if ortho:\n                raw=Path(ortho).read_bytes()\n                if len(raw)>12*1024*1024 or raw[:8]!=b\'\\x89PNG\\r\\n\\x1a\\n\' or len(raw)<24 or struct.unpack(\'>II\',raw[16:24])!=(z.shape[1],z.shape[0]):\n                    raise ValueError(\'Ortofoto preparata non coerente con la griglia.\')\n', 'forecast_service/sectors.py': '"""Optional beach subdivisions; forcing belongs to the parent location."""\nimport math\nimport re\n\n\ndef valid_id(value):\n    return isinstance(value, str) and re.fullmatch(r\'[a-z0-9]+(?:-[a-z0-9]+)*\', value) is not None\n\n\ndef validate_thresholds(thresholds):\n    if not isinstance(thresholds, list) or not thresholds:\n        raise ValueError(\'Soglie del settore mancanti.\')\n    try:\n        ids = [t[\'id\'] for t in thresholds]\n        heights = [t[\'elevation_m\'] for t in thresholds]\n        if len(set(ids)) != len(ids) or any(not valid_id(i.replace(\'_\', \'-\')) for i in ids):\n            raise ValueError(\'Identificativi delle soglie non validi.\')\n        if any(not isinstance(h, (int, float)) or not math.isfinite(h) for h in heights) or heights != sorted(set(heights)):\n            raise ValueError(\'Quote delle soglie non valide.\')\n        if any(t[\'season\'] not in (\'summer\', \'all_year\') or not t.get(\'description\') for t in thresholds):\n            raise ValueError(\'Descrizione o stagione delle soglie non valida.\')\n    except (KeyError, TypeError) as error:\n        raise ValueError(\'Soglie del settore non valide.\') from error\n\n\ndef resolve_sectors(location, thresholds):\n    """Derive slopes from assigned transects, never from their order in a list."""\n    from forecast_service.exposure import validate_exposure\n    sectors = location.get(\'sectors\', {})\n    if not isinstance(sectors, dict):\n        raise ValueError(\'I settori devono essere un dizionario.\')\n    transects = location.get(\'transects\') or []\n    by_id = {}\n    for row in transects:\n        key = str(row[\'transect_id\'])\n        if key in by_id:\n            raise ValueError(\'Identificativo transetto duplicato.\')\n        by_id[key] = row\n    assigned = set()\n    resolved = {}\n    allowed = {\'name\', \'transect_ids\', \'thresholds\', \'wave_exposure\', \'mapping\', \'notes\', \'summer_season\'}\n    for key, sector in sectors.items():\n        if not valid_id(key) or key == \'whole\' or not isinstance(sector, dict) or not sector.get(\'name\'):\n            raise ValueError(\'Identificativo o nome del settore non valido (whole è riservato).\')\n        if set(sector) - allowed:\n            raise ValueError(\'Campi del settore non ammessi: \' + \', \'.join(sorted(set(sector)-allowed)))\n        if not isinstance(sector.get(\'transect_ids\'), list):\n            raise ValueError(\'transect_ids deve essere una lista di identificativi.\')\n        ids = [str(i) for i in sector[\'transect_ids\']]\n        if not ids or len(set(ids)) != len(ids) or set(ids)-by_id.keys():\n            raise ValueError(f\'Transetti mancanti, duplicati o sconosciuti nel settore {key}.\')\n        if assigned.intersection(ids):\n            raise ValueError(\'Uno stesso transetto è assegnato a più settori.\')\n        assigned.update(ids)\n        selected = [by_id[i] for i in ids]\n        slopes = [row.get(\'slope\') for row in selected]\n        if any(row.get(\'status\', \'valid\') != \'valid\' for row in selected) or not slopes or any(isinstance(s, bool) or not isinstance(s, (int, float)) or not math.isfinite(s) or not 0 < s < 1 for s in slopes):\n            raise ValueError(f\'Pendenze dei transetti non valide nel settore {key}.\')\n        slopes = sorted(set(slopes))\n        merged = {k: v for k, v in location.items() if k not in (\'sectors\', \'mapping\')}\n        merged.update(sector, slopes=slopes, transects=selected, thresholds=sector.get(\'thresholds\', thresholds))\n        validate_thresholds(merged[\'thresholds\'])\n        validate_exposure(merged.get(\'wave_exposure\'))\n        season = merged.get(\'summer_season\')\n        if season:\n            from datetime import datetime\n            dates = [season.get(\'start_month_day\'), season.get(\'end_month_day\')]\n            if any(dates) and not all(dates):\n                raise ValueError(\'Definire entrambi gli estremi della stagione.\')\n            for value in dates:\n                if value:\n                    datetime.strptime(\'2000-\'+value, \'%Y-%m-%d\')\n        resolved[key] = merged\n    if sectors and assigned != set(by_id):\n        raise ValueError(\'Ogni transetto incluso deve appartenere a un settore: \' + \', \'.join(sorted(set(by_id)-assigned)))\n    return resolved\n\n\ndef mapping_entries(location):\n    """Stable identifiers for optional per-beach/per-sector cartography."""\n    if location.get(\'mapping\'):\n        yield \'whole\', location[\'mapping\']\n    for key, sector in (location.get(\'sectors\') or {}).items():\n        if sector.get(\'mapping\'):\n            yield key, sector[\'mapping\']\n\n\ndef validate_mappings(location, slug):\n    for key, mapping in mapping_entries(location):\n        prefix = f\'config/study_sites/{slug}/cartography/{key}/\'\n        if not isinstance(mapping, dict) or mapping.get(\'grid_file\') != prefix+\'grid.npz\':\n            raise ValueError(\'Percorso della griglia cartografica non valido.\')\n        if mapping.get(\'orthophoto_file\') not in (None, prefix+\'orthophoto.png\'):\n            raise ValueError(\'Percorso dell’ortofoto non valido.\')\n        for field in (\'survey_credit\', \'survey_date\', \'license\', \'vertical_reference\'):\n            if not isinstance(mapping.get(field), str) or not mapping[field].strip():\n                raise ValueError(f\'Cartografia: compilare {field}.\')\n        if mapping.get(\'vertical_alignment_confirmed\') is not True:\n            raise ValueError(\'Confermare il raccordo verticale DEM/livello marino prima di attivare la mappa.\')\n        if not isinstance(mapping.get(\'dem_offset_m\'), (int, float)) or not math.isfinite(mapping[\'dem_offset_m\']):\n            raise ValueError(\'Correzione verticale del DEM non valida.\')\n', 'forecast_service/inundation.py': '"""Static, sea-connected potential reach. Not hydrodynamics or flood depth.\n\nPrepared grids are bounded, north-up EPSG:3857 rasters (bounds in WGS84). Flood the full grid\nbefore clipping to the display polygon; sector edges are not artificial walls.\nThe one-off raster preparation lives separately to keep production lightweight.\n"""\nfrom collections import deque\nimport hashlib\nimport math\nfrom pathlib import Path\nimport struct\nimport zipfile\nimport zlib\n\nimport numpy as np\n\nMAX_CELLS = 1024 * 1024\nMETHOD = \'connected_static_twl_filtered_p75_p95_v1\'\n\n\ndef png_bytes(rgba):\n    rgba = np.asarray(rgba, dtype=np.uint8)\n    height, width, channels = rgba.shape\n    if channels != 4 or height * width > MAX_CELLS:\n        raise ValueError(\'Dimensioni PNG non valide.\')\n    def chunk(kind, data):\n        return struct.pack(\'>I\', len(data)) + kind + data + struct.pack(\'>I\', zlib.crc32(kind+data) & 0xffffffff)\n    pixels = b\'\'.join(b\'\\0\'+row.tobytes() for row in rgba)\n    return b\'\\x89PNG\\r\\n\\x1a\\n\' + chunk(b\'IHDR\', struct.pack(\'>IIBBBBB\', width, height, 8, 6, 0, 0, 0)) + chunk(b\'IDAT\', zlib.compress(pixels)) + chunk(b\'IEND\', b\'\')\n\n\ndef load_grid(path):\n    # Bound compressed-array expansion before numpy allocates anything.\n    with zipfile.ZipFile(path) as archive:\n        if len(archive.namelist()) != 4 or set(archive.namelist()) != {\'z.npy\', \'seeds.npy\', \'display_mask.npy\', \'bounds.npy\'} or sum(m.file_size for m in archive.infolist()) > 16*1024*1024:\n            raise ValueError(\'Griglia cartografica non valida o troppo grande.\')\n    with np.load(path, allow_pickle=False) as data:\n        z = data[\'z\']; seeds = data[\'seeds\']; display = data[\'display_mask\']; bounds = data[\'bounds\']\n    if z.ndim != 2 or not z.size or z.size > MAX_CELLS or max(z.shape) > 1024 or z.dtype.kind != \'f\' or np.isinf(z).any():\n        raise ValueError(\'DEM preparato non valido.\')\n    if seeds.shape != z.shape or display.shape != z.shape or seeds.dtype != bool or display.dtype != bool:\n        raise ValueError(\'Maschere del DEM non coerenti.\')\n    if not seeds.any() or not np.isfinite(z[seeds]).all() or not display.any():\n        raise ValueError(\'Mancano celle valide di partenza verso mare o area da rappresentare.\')\n    if bounds.shape != (4,) or not np.isfinite(bounds).all():\n        raise ValueError(\'Estensione geografica non valida.\')\n    west, south, east, north = bounds.tolist()\n    if not -180 <= west < east <= 180 or not -90 <= south < north <= 90:\n        raise ValueError(\'Estensione geografica non valida.\')\n    return z, seeds, display, [[south, west], [north, east]]\n\n\ndef connected_mask(z, seeds, level):\n    """Eight-neighbour connectivity, as r.lake\'s 3x3 neighbourhood."""\n    if not math.isfinite(level):\n        raise ValueError(\'Quota di propagazione non valida.\')\n    eligible = np.isfinite(z) & (z < level)\n    filled = seeds & eligible\n    queue = deque(zip(*np.nonzero(filled)))\n    height, width = z.shape\n    while queue:\n        row, col = queue.popleft()\n        for r in range(max(0, row-1), min(height, row+2)):\n            for c in range(max(0, col-1), min(width, col+2)):\n                if eligible[r, c] and not filled[r, c]:\n                    filled[r, c] = True\n                    queue.append((r, c))\n    return filled\n\n\ndef build_map(unit, bulletin, mapping, root, slug, unit_id):\n    from forecast_service.ensemble import STATISTICS_METHOD\n    from forecast_service.exposure import is_exposed\n    from forecast_service.time_utils import parse_time\n    start = parse_time(bulletin.get(\'display_from\', bulletin[\'processing_started_at\'])).replace(minute=0, second=0, microsecond=0)\n    from datetime import timedelta\n    end = start + timedelta(hours=bulletin[\'display_hours\'])\n    if bulletin.get(\'display_to\'):\n        end = parse_time(bulletin[\'display_to\'])\n    rows = [r for r in unit[\'hourly\'] if start <= parse_time(r[\'time\']) < end]\n    expected = [start+timedelta(hours=i) for i in range(bulletin[\'display_hours\'])]\n    if [parse_time(r[\'time\']) for r in rows] != expected or any(r.get(\'statistics_method\') != STATISTICS_METHOD or not all(isinstance(r.get(f\'twl_p{q}_m\'), (int, float)) and math.isfinite(r[f\'twl_p{q}_m\']) for q in (75, 95)) for r in rows):\n        raise ValueError(\'La mappa richiede tutte le 96 ore e percentili filtrati validi.\')\n    grid_path = Path(root)/mapping[\'grid_file\']\n    z, seeds, display, bounds = load_grid(grid_path)\n    prefix = f\'maps/{slug}/{unit_id}\'\n    assets = {}\n    layers = []\n    for q, color in [(75, (35, 129, 188, 115)), (95, (107, 197, 232, 90))]:\n        peak = max(rows, key=lambda r: r[f\'twl_p{q}_m\'])\n        level = peak[f\'twl_p{q}_m\']\n        if not (z[seeds] < level).any():\n            raise ValueError(\'Quota prevista sotto tutte le celle di partenza: spostare i punti di avvio verso mare nel DEM.\')\n        mask = connected_mask(z, seeds, level) & display\n        rgba = np.zeros((*z.shape, 4), dtype=np.uint8)\n        rgba[mask] = color\n        filename = f\'{prefix}-p{q}.png\'\n        assets[filename] = png_bytes(rgba)\n        layers.append({\'percentile\': q, \'level_m\': level, \'peak_time\': peak[\'time\'], \'image\': \'data/\'+filename, \'cells\': int(mask.sum())})\n    ortho = None\n    if mapping.get(\'orthophoto_file\'):\n        image = (Path(root)/mapping[\'orthophoto_file\']).read_bytes()\n        if len(image) > 12*1024*1024 or image[:8] != b\'\\x89PNG\\r\\n\\x1a\\n\' or len(image) < 24 or struct.unpack(\'>II\', image[16:24]) != (z.shape[1], z.shape[0]):\n            raise ValueError(\'Ortofoto preparata non coerente con il DEM.\')\n        filename = prefix+\'-orthophoto.png\'; assets[filename] = image; ortho = \'data/\'+filename\n    exposure = [is_exposed(r.get(\'direction_deg\'), unit.get(\'wave_exposure\')) if r.get(\'hs_m\', 0) > 0 else None for r in rows]\n    return {\'status\': \'available\', \'method\': METHOD, \'data_mode\': bulletin[\'mode\'], \'period_from\': start.isoformat(), \'period_to\': end.isoformat(),\n            \'bounds\': bounds, \'layers\': layers, \'orthophoto\': ortho,\n            \'survey_credit\': mapping[\'survey_credit\'], \'survey_date\': mapping[\'survey_date\'], \'license\': mapping[\'license\'],\n            \'vertical_reference\': mapping[\'vertical_reference\'], \'dem_offset_m\': mapping[\'dem_offset_m\'],\n            \'grid_sha256\': hashlib.sha256(grid_path.read_bytes()).hexdigest(),\n            \'hours_non_frontal\': exposure.count(False), \'hours_direction_unknown\': exposure.count(None),\n            \'interpretation\': \'Estensione statica potenziale, non profondità né probabilità di allagamento. Il runup è intermittente; nessuna trasformazione direzionale o simulazione di sormonto, portate e durata.\'}, assets\n\n\ndef write_maps(bulletin, output, root):\n    """An unavailable optional map must not suppress the otherwise valid forecast."""\n    for slug, location in bulletin[\'locations\'].items():\n        for unit_id, unit in [(\'whole\', location), *location.get(\'sectors\', {}).items()]:\n            mapping = unit.get(\'mapping\')\n            if not mapping:\n                continue\n            try:\n                metadata, assets = build_map(unit, bulletin, mapping, root, slug, unit_id)\n            except (ValueError, OSError, KeyError, zipfile.BadZipFile) as error:\n                unit[\'inundation_map\'] = {\'status\': \'unavailable\', \'reason\': str(error)}\n                continue\n            for name, raw in assets.items():\n                destination = Path(output)/name\n                destination.parent.mkdir(parents=True, exist_ok=True)\n                destination.write_bytes(raw)\n            unit[\'inundation_map\'] = metadata\n', 'forecast_service/mapping_setup.py': '"""One-off GeoTIFF preparation for lightweight static-map publication."""\nfrom pathlib import Path\nimport json\n\nimport numpy as np\n\nfrom forecast_service.inundation import png_bytes, load_grid\nfrom forecast_service.sectors import valid_id, validate_mappings\n\n\ndef prepare_mapping(dem_path, output, *, slug, unit_id=\'whole\', orthophoto_path=None,\n                    display_polygon=None, seed_points=None, survey_credit=\'\', survey_date=\'\',\n                    license=\'\', vertical_reference=\'\', dem_offset_m=0.0,\n                    vertical_alignment_confirmed=False, max_dimension=768):\n    """Reproject an already clipped local DEM; preserve original files outside Git.\n\n    Polygon and sea-side seed points use WGS84 lon/lat. Offset is ADDED once to\n    DEM heights to match the TWL reference, independently of profile corrections.\n    Full DEM extent is the connectivity domain, polygon is only a display mask.\n    """\n    import rasterio\n    from rasterio.warp import calculate_default_transform, reproject, Resampling, transform_bounds, transform_geom, transform as reproject_coordinates\n    from rasterio.transform import from_bounds, rowcol\n    from rasterio.features import geometry_mask\n    if not valid_id(slug) or not valid_id(unit_id) or not isinstance(max_dimension, int) or not 32 <= max_dimension <= 1024:\n        raise ValueError(\'Identificativi o dimensione della griglia non validi.\')\n    prefix = f\'config/study_sites/{slug}/cartography/{unit_id}/\'\n    metadata = dict(grid_file=prefix+\'grid.npz\', orthophoto_file=prefix+\'orthophoto.png\' if orthophoto_path else None,\n                    survey_credit=survey_credit, survey_date=survey_date, license=license, vertical_reference=vertical_reference,\n                    dem_offset_m=float(dem_offset_m), vertical_alignment_confirmed=vertical_alignment_confirmed)\n    validate_mappings({\'mapping\': metadata} if unit_id == \'whole\' else {\'sectors\': {unit_id: {\'mapping\': metadata}}}, slug)\n    if not seed_points:\n        raise ValueError(\'Indicare punti di partenza verso mare in celle valide del DEM (lon, lat).\')\n    with rasterio.open(dem_path) as dem:\n        if not dem.crs or dem.count != 1:\n            raise ValueError(\'DEM: GeoTIFF con una banda, CRS e quote in metri richiesti.\')\n        if dem.units[0] not in (None, \'\', \'m\', \'metre\', \'meter\', \'metres\', \'meters\'):\n            raise ValueError(\'Convertire le quote del DEM in metri.\')\n        west, south, east, north = transform_bounds(dem.crs, \'EPSG:4326\', *dem.bounds)\n        if not -180 <= west < east <= 180 or not -90 <= south < north <= 90 or east-west > .2 or north-south > .2:\n            raise ValueError(\'Ritagliare prima un DEM locale (estensione massima 0,2° per asse).\')\n        left, bottom, right, top = transform_bounds(dem.crs, \'EPSG:3857\', *dem.bounds)\n        _, width, height = calculate_default_transform(dem.crs, \'EPSG:3857\', dem.width, dem.height, *dem.bounds)\n        factor = min(1, max_dimension/max(width, height))\n        width, height = max(1, round(width*factor)), max(1, round(height*factor))\n        transform = from_bounds(left, bottom, right, top, width, height)\n        z = np.full((height, width), np.nan, dtype=np.float32)\n        # Never average away a seawall. Resolution limits still need field review.\n        reproject(rasterio.band(dem, 1), z, src_transform=dem.transform, src_crs=dem.crs,\n                  src_nodata=dem.nodata, dst_transform=transform, dst_crs=\'EPSG:3857\', dst_nodata=np.nan,\n                  resampling=Resampling.nearest)\n        z += metadata[\'dem_offset_m\']\n    if not display_polygon or display_polygon.get(\'type\') not in (\'Polygon\', \'MultiPolygon\'):\n        raise ValueError(\'Definire il poligono WGS84 dell’area da rappresentare.\')\n    display = geometry_mask([transform_geom("EPSG:4326", "EPSG:3857", display_polygon)], out_shape=z.shape, transform=transform, invert=True)\n    seeds = np.zeros(z.shape, dtype=bool)\n    for point in seed_points:\n        if len(point) != 2 or not np.isfinite(point).all():\n            raise ValueError(\'Coordinate dei punti a mare non valide.\')\n        xx, yy = reproject_coordinates("EPSG:4326", "EPSG:3857", [point[0]], [point[1]])\n        row, col = rowcol(transform, xx[0], yy[0])\n        if not 0 <= row < height or not 0 <= col < width or not np.isfinite(z[row, col]):\n            raise ValueError(\'Un punto a mare è fuori dal DEM o su NoData: scegliere una cella valida.\')\n        seeds[row, col] = True\n    output = Path(output); output.mkdir(parents=True, exist_ok=True)\n    grid = output/\'grid.npz\'\n    np.savez_compressed(grid, z=z, seeds=seeds, display_mask=display, bounds=np.array([west, south, east, north]))\n    load_grid(grid)\n    assets = {f\'cartography/{unit_id}/grid.npz\': grid}\n    if orthophoto_path:\n        with rasterio.open(orthophoto_path) as ortho:\n            if not ortho.crs or ortho.count not in (1, 3, 4) or any(dtype != \'uint8\' for dtype in ortho.dtypes):\n                raise ValueError(\'Ortofoto: GeoTIFF georeferenziato a 8 bit, grigio/RGB/RGBA.\')\n            rgba = np.zeros((height, width, 4), dtype=np.uint8)\n            bands = [1, 1, 1] if ortho.count == 1 else [1, 2, 3]\n            for channel, band in enumerate(bands):\n                reproject(rasterio.band(ortho, band), rgba[:, :, channel], src_transform=ortho.transform,\n                          src_crs=ortho.crs, dst_transform=transform, dst_crs=\'EPSG:3857\', resampling=Resampling.bilinear)\n            valid = np.zeros((height, width), dtype=np.uint8)\n            # Source mask honours alpha and NoData, without reading the whole mosaic.\n            from rasterio.vrt import WarpedVRT\n            with WarpedVRT(ortho, crs=\'EPSG:3857\', transform=transform, width=width, height=height) as vrt:\n                valid = vrt.dataset_mask()\n            rgba[:, :, 3] = valid\n            image = output/\'orthophoto.png\'; image.write_bytes(png_bytes(rgba))\n            assets[f\'cartography/{unit_id}/orthophoto.png\'] = image\n    metadata[\'prepared_crs\'] = \'EPSG:3857\'\n    metadata[\'prepared_grid_shape\'] = list(z.shape)\n    metadata[\'dem_source_name\'] = Path(dem_path).name\n    metadata[\'orthophoto_source_name\'] = Path(orthophoto_path).name if orthophoto_path else None\n    metadata[\'processing_note\'] = \'DEM ricampionato al vicino più prossimo; verificare opere strette e risoluzione. Connessione sul DEM completo, ritaglio sul poligono solo in uscita.\'\n    (output/\'mapping.json\').write_text(json.dumps(metadata, ensure_ascii=False, indent=2)+\'\\n\')\n    return metadata, assets\n', 'forecast_service/ensemble.py': '"""Statistical screening of model/slope results; scientific model outputs stay raw."""\nfrom math import isfinite\nfrom statistics import fmean, pstdev\n\nSTATISTICS_METHOD = \'sigma_clip_3_population_v1\'\nSTATISTICS_METADATA = {\n    \'method\': STATISTICS_METHOD,\n    \'grouping\': \'location_or_sector_hour\',\n    \'ensemble_unit\': \'model_slope\',\n    \'sigma_multiplier\': 3,\n    \'sigma_ddof\': 0,\n    \'passes\': 1,\n    \'bounds_inclusive\': True,\n    \'raw_csv_preserved\': True,\n}\n\n\ndef sigma_clip(values):\n    """One pass, population sigma; values exactly at mean ±3σ are retained."""\n    values = list(values)\n    if any(not isfinite(value) for value in values):\n        raise ValueError(\'Il filtro richiede risultati finiti.\')\n    if not values:\n        return [], None, None\n    mean = fmean(values)\n    sigma = pstdev(values, mu=mean)\n    included = [True] * len(values) if sigma == 0 else [abs(value-mean) <= 3*sigma for value in values]\n    return included, mean, sigma\n', 'forecast_service/LICENSE-wave-climate.txt': 'MIT License\n\nCopyright (c) 2026 Alessio Rovere\n\nPermission is hereby granted, free of charge, to any person obtaining a copy\nof this software and associated documentation files (the "Software"), to deal\nin the Software without restriction, including without limitation the rights\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\ncopies of the Software, and to permit persons to whom the Software is\nfurnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in all\ncopies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE\nSOFTWARE.\n', 'config/study_sites/borghetto-santo-spirito/sea_level_reference_2025.json': '{\n  "schema_version": 1,\n  "kind": "copernicus_historical_mean",\n  "year": 2025,\n  "dataset_id": "cmems_mod_med_phy-ssh_anfc_4.2km-2D_PT1H-m",\n  "dataset_version": "202511",\n  "variable": "zos",\n  "requested_point": {\n    "latitude": 44.08,\n    "longitude": 8.33\n  },\n  "selected_point": {\n    "latitude": 44.0625,\n    "longitude": 8.333333015441895\n  },\n  "start": "2025-01-01T00:00:00Z",\n  "end": "2025-12-31T23:00:00Z",\n  "n_records": 8760,\n  "mean_ssh_m": -0.4032032064388128,\n  "monthly": {\n    "1": {\n      "mean_ssh_m": -0.410685404639785,\n      "n_records": 744\n    },\n    "2": {\n      "mean_ssh_m": -0.5115411359226191,\n      "n_records": 672\n    },\n    "3": {\n      "mean_ssh_m": -0.4807945041263441,\n      "n_records": 744\n    },\n    "4": {\n      "mean_ssh_m": -0.46933154022222223,\n      "n_records": 720\n    },\n    "5": {\n      "mean_ssh_m": -0.4296025681586021,\n      "n_records": 744\n    },\n    "6": {\n      "mean_ssh_m": -0.4312678409444445,\n      "n_records": 720\n    },\n    "7": {\n      "mean_ssh_m": -0.3726959052822581,\n      "n_records": 744\n    },\n    "8": {\n      "mean_ssh_m": -0.3369342917688172,\n      "n_records": 744\n    },\n    "9": {\n      "mean_ssh_m": -0.3413011670013889,\n      "n_records": 720\n    },\n    "10": {\n      "mean_ssh_m": -0.36464155797715053,\n      "n_records": 744\n    },\n    "11": {\n      "mean_ssh_m": -0.31705964612500004,\n      "n_records": 720\n    },\n    "12": {\n      "mean_ssh_m": -0.38133004919354835,\n      "n_records": 744\n    }\n  },\n  "data_mode": "study",\n  "local_datum_status": "not_validated",\n  "method": "Arithmetic mean of all hourly SSH values in one complete UTC calendar year; fixed for future forecasts.",\n  "input_sha256": "c5f2271ef0e2de0c95d91b074a032294af6ee9c1e46074d74c122dfacf6eaef2"\n}\n'}
for relative, source in BUNDLED_SOURCES.items():
    target = RUNTIME / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source, encoding="utf-8")
for module_name in list(sys.modules):
    if module_name=="forecast_service" or module_name.startswith("forecast_service."):
        del sys.modules[module_name]
sys.path.insert(0, str(RUNTIME))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import folium
from IPython.display import display, FileLink
from pyproj import CRS, Transformer
from forecast_service.site_setup import analyze_transects, BEACHFACE_METHOD_VERSION
from forecast_service.era5_download import download_series
from forecast_service.climate import build_climate
from forecast_service.exposure import parse_exposure
from forecast_service.extremes import build_directional_extremes
from forecast_service.sealevel_reference import download_reference, validate_reference, apply_reference
from forecast_service.site_package import write_site_package
print("Metodo pendenze:", BEACHFACE_METHOD_VERSION)

In [ ]:
#@title Modalità e salvataggio
DEMO = False #@param {type:"boolean"}
USA_GOOGLE_DRIVE = True #@param {type:"boolean"}
CARTELLA_DRIVE = 'OndeARiva_setup' #@param {type:"string"}
if USA_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive") / CARTELLA_DRIVE
else:
    BASE = Path("/content/ondeariva_dati") if Path("/content").exists() else RUNTIME / "dati"
BASE.mkdir(parents=True, exist_ok=True)
print("Modalità:", "DIMOSTRATIVA — profili e onde sintetici" if DEMO else "SITO DI STUDIO — dati dell’utente")
print("Cartella:", BASE)
print("La cartella del runtime è temporanea. Per conservare la cache usa Drive oppure scarica il pacchetto finale.")

## 2. Definisci il sito
Le coordinate sono in gradi WGS84. Il punto sulla mappa identifica il sito; quello offshore serve a scaricare onde Copernicus ed ERA5. La cella marina ERA5 verrà verificata durante il download. Il punto del livello marino può essere distinto da quello delle onde.

I valori iniziali sono quelli provvisori di Borghetto; possono essere sostituiti liberamente. Il campo `ID_SITO` deve usare lettere minuscole, numeri e trattini.

In [ ]:
#@title Identità e punti di estrazione
ID_SITO = 'borghetto-santo-spirito' #@param {type:"string"}
NOME_SITO = 'Borghetto Santo Spirito (SV)' #@param {type:"string"}
TRATTO_DI_COSTA = 'Riviera di Ponente · Mar Ligure' #@param {type:"string"}
LAT_SITO = 44.109 #@param {type:"number"}
LON_SITO = 8.242 #@param {type:"number"}
LAT_ONDE = 44.08 #@param {type:"number"}
LON_ONDE = 8.33 #@param {type:"number"}
LIVELLO_STESSO_PUNTO = True #@param {type:"boolean"}
LAT_LIVELLO = 44.08 #@param {type:"number"}
LON_LIVELLO = 8.33 #@param {type:"number"}
import re
if not re.fullmatch(r"[a-z0-9]+(?:-[a-z0-9]+)*", ID_SITO):
    raise ValueError("ID_SITO non valido.")
for lat, lon in [(LAT_SITO,LON_SITO),(LAT_ONDE,LON_ONDE),(LAT_LIVELLO,LON_LIVELLO)]:
    if not np.isfinite([lat,lon]).all() or not 30<=lat<=46.5 or not -18.2<=lon<=37:
        raise ValueError("Le coordinate devono essere nel dominio mediterraneo.")
WORK = BASE / ID_SITO
WORK.mkdir(parents=True, exist_ok=True)
SITE_INPUT = dict(name=NOME_SITO, area=TRATTO_DI_COSTA,
    map_point=dict(latitude=LAT_SITO,longitude=LON_SITO),
    wave_point=dict(latitude=LAT_ONDE,longitude=LON_ONDE))
MAPPA = folium.Map(location=[LAT_SITO,LON_SITO],zoom_start=12,tiles="OpenStreetMap")
folium.Marker([LAT_SITO,LON_SITO],tooltip="Sito costiero").add_to(MAPPA)
folium.CircleMarker([LAT_ONDE,LON_ONDE],radius=7,color="#006d77",tooltip="Punto onde richiesto").add_to(MAPPA)
folium.LatLngPopup().add_to(MAPPA)
display(MAPPA)
print("Clicca sulla mappa per leggere coordinate, poi riportale nei campi e riesegui questa cella.")

### Esposizione della spiaggia alle onde
Indica le **direzioni da cui provengono le onde**, non quelle verso cui si muovono: 0° = Nord, 90° = Est, 180° = Sud, 270° = Ovest, in senso orario rispetto al Nord geografico. I settori devono riferirsi al punto offshore scelto e alla spiaggia rappresentata, tenendo conto di costa, promontori e opere.

Scrivi coppie `da,a`, separate da `;` per più settori. Esempi solo di formato: `90,180` comprende Est→Sud; `315,45` attraversa il Nord; `0,360` include tutte le direzioni. Gli estremi sono inclusi. **Lascia vuoto finché non hai definito l’esposizione del sito**: nessun settore di esempio viene applicato automaticamente.

Il portale avverte quando la direzione al largo è fuori dai settori indicati. Questa verifica non simula rifrazione, diffrazione o riparo e non azzera né corregge automaticamente onde, runup o TWL. Anche il confronto storico ERA5 resta riferito a tutte le direzioni.

In [ ]:
#@title Settori di provenienza delle onde che espongono la spiaggia
SETTORI_PROVENIENZA = '' #@param {type:"string"}
NOTE_ESPOSIZIONE = '' #@param {type:"string"}
ESPOSIZIONE = parse_exposure(SETTORI_PROVENIENZA, NOTE_ESPOSIZIONE)
if ESPOSIZIONE is None:
    print("Esposizione non definita: il portale mostrerà un’avvertenza.")
else:
    display(pd.DataFrame(ESPOSIZIONE["sectors_deg"], columns=["Da (gradi)", "A (gradi, senso orario)"]))
    fig, ax = plt.subplots(figsize=(5, 5), subplot_kw={"projection": "polar"})
    ax.set_theta_zero_location("N"); ax.set_theta_direction(-1)
    ax.set_xticks(np.deg2rad(np.arange(0, 360, 45)), ["N", "NE", "E", "SE", "S", "SO", "O", "NO"])
    ax.set_yticks([])
    for start, end in ESPOSIZIONE["sectors_deg"]:
        span = 360 if end-start == 360 else (end-start) % 360
        angles = np.deg2rad(np.linspace(start, start+span, 181))
        ax.fill_between(angles, 0, 1, color="#527d78", alpha=.35)
    ax.set_title("Direzioni di provenienza che espongono la spiaggia")
    plt.show()
    print("Schema orientativo: non è una simulazione della propagazione verso riva.")

## 3. Carica i profili esportati da QGIS
Usa il CSV del tuo notebook, con colonne predefinite `TR_ID`, `distance`, `SAMPLE_1`, `X`, `Y`. Puoi modificare i nomi sotto. **X e Y devono appartenere a un CRS proiettato con unità in metri**, non essere longitudine e latitudine. `distance` serve a ordinare i punti; la distanza della regressione viene ricalcolata da X/Y come nel notebook fornito.

In modalità demo vengono creati due transetti sintetici, uno con ordine inverso, con un abbassamento del terreno dietro la prima quota sopra `Z_MAX`: quel tratto deve rimanere escluso dalla stima.

In [ ]:
#@title Colonne, coordinate e profili
COL_ID = 'TR_ID' #@param {type:"string"}
COL_DIST = 'distance' #@param {type:"string"}
COL_Z = 'SAMPLE_1' #@param {type:"string"}
COL_X = 'X' #@param {type:"string"}
COL_Y = 'Y' #@param {type:"string"}
CRS_PROFILI = 'EPSG:32632' #@param {type:"string"}
FILE_PROFILI = '' #@param {type:"string"}
SEPARATORE = ',' #@param {type:"string"}
DECIMALE = '.' #@param {type:"string"}
CORREZIONE_QUOTE_DEM_M = 0.0 #@param {type:"number"}
RIFERIMENTO_QUOTE = 'Livello medio mare, da verificare' #@param {type:"string"}
crs = CRS.from_user_input(CRS_PROFILI)
if not crs.is_projected or not all(abs(a.unit_conversion_factor-1)<1e-9 for a in crs.axis_info[:2]):
    raise ValueError("Usa coordinate X/Y in un CRS proiettato in metri.")
if DEMO:
    to_xy = Transformer.from_crs(4326,crs,always_xy=True)
    x0,y0=to_xy.transform(LON_SITO,LAT_SITO)
    demo_rows=[]
    for name,beta,reverse in [("T1",.045,False),("T2",.065,True)]:
        d=np.arange(45)*2.0
        z=beta*d-.3
        z[(d>=40)&(d<=56)]=.4
        xy=np.column_stack([x0+.6*d,y0+.8*d+(10 if name=="T2" else 0)])
        if reverse:
            z=z[::-1]; xy=xy[::-1]
        for i,dist in enumerate(d):
            demo_rows.append({COL_ID:name,COL_DIST:dist,COL_Z:z[i],COL_X:xy[i,0],COL_Y:xy[i,1]})
    profili=pd.DataFrame(demo_rows)
    PROFILE_SOURCE="profili_sintetici"
else:
    if not FILE_PROFILI:
        from google.colab import files
        uploaded=files.upload()
        if len(uploaded)!=1:
            raise ValueError("Carica un solo CSV di profili.")
        name,content=next(iter(uploaded.items()))
        source=WORK/Path(name).name
        source.write_bytes(content)
    else:
        source=Path(FILE_PROFILI)
    profili=pd.read_csv(source,sep=SEPARATORE,decimal=DECIMALE)
    PROFILE_SOURCE=source.name
if len({COL_ID,COL_DIST,COL_Z,COL_X,COL_Y})!=5:
    raise ValueError("I nomi delle cinque colonne devono essere distinti.")
PROFILE_DATA_MODE="demo" if DEMO else "study"
profili[COL_Z]=pd.to_numeric(profili[COL_Z],errors="raise")+CORREZIONE_QUOTE_DEM_M
print(f"{len(profili)} punti, {profili[COL_ID].nunique()} transetti. Fonte: {PROFILE_SOURCE}")
display(profili.head())

## 4. Calcola la pendenza come nel notebook di riferimento
La procedura:
1. elimina le sole quote mancanti e ordina per `distance`;
2. se la media delle prime 10 quote è maggiore di quella delle ultime 10, inverte il profilo;
3. calcola `x_mare = hypot(X-X_primo, Y-Y_primo)`;
4. si ferma **prima del primo punto con quota maggiore di `Z_MAX`**;
5. fra i punti precedenti usa quelli con quota almeno `Z_MIN` e calcola `np.polyfit(x_mare, quota, 1)[0]`.

La pendenza resta con il proprio segno: un risultato negativo non viene convertito in positivo. Vengono segnalati i profili degeneri o con pendenza fuori da 0–1. Con profili molto corti, le prime/ultime 10 quote possono sovrapporsi: verifica il grafico e, se necessario, scegli esplicitamente l’orientamento.

In [ ]:
#@title Intervallo della battigia e orientamento
Z_MIN = 0.0 #@param {type:"number"}
Z_MAX = 1.0 #@param {type:"number"}
ORIENTAMENTO = 'auto' #@param ["auto", "distance_increases_landward", "distance_increases_seaward"]
riepilogo, preparati, selezioni = analyze_transects(profili,Z_MIN,Z_MAX,
    id_col=COL_ID,distance_col=COL_DIST,elevation_col=COL_Z,x_col=COL_X,y_col=COL_Y,orientation=ORIENTAMENTO)
display(riepilogo)
ANALYSIS_SIGNATURE=(Z_MIN,Z_MAX,ORIENTAMENTO,COL_ID,COL_DIST,COL_Z,COL_X,COL_Y,
    hashlib.sha256(pd.util.hash_pandas_object(profili,index=True).values.tobytes()).hexdigest())
riepilogo.to_csv(WORK/"pendenze_transetti.csv",index=False)
validi=riepilogo[riepilogo.status=="valid"]
if validi.empty:
    raise ValueError("Nessun transetto valido: controlla colonne, intervallo e orientamento.")
print("Pendenze valide:",validi.slope.min(),"—",validi.slope.max(),"m/m")
if validi.orientation_ambiguous.any():
    print("Attenzione: orientamento automatico ambiguo per alcuni profili; controlla i grafici.")

In [ ]:
#@title Verifica un transetto nel grafico
ID_TRANSETTO_DA_VEDERE = '3' #@param {type:"string"}
identifier=ID_TRANSETTO_DA_VEDERE or str(validi.iloc[0].transect_id)
if identifier not in preparati:
    raise ValueError("Transetto non valido o inesistente; scegli un ID dalla tabella.")
p=preparati[identifier]; sel=selezioni[identifier]
info=validi[validi.transect_id==identifier].iloc[0]
fig,ax=plt.subplots(figsize=(10,4))
ax.plot(p.x_mare,p[COL_Z],color=".35",label="Profilo completo")
ax.scatter(p.loc[sel,"x_mare"],p.loc[sel,COL_Z],color="#d87832",label="Punti usati nella regressione",zorder=3)
xfit=p.loc[sel,"x_mare"]
ax.plot(xfit,info.slope*xfit+info.fit_intercept_m,color="#b54135",ls="--",label="Retta stimata")
ax.axhspan(Z_MIN,Z_MAX,color="#d5e9ee",alpha=.35,label="Intervallo di quota")
if info.first_above_zmax_found:
    ax.axvline(p.loc[int(info.stop_index),"x_mare"],color="#8b633f",ls=":",label="Primo punto sopra Z_MAX")
ax.set(xlabel="Distanza euclidea dal primo punto a mare [m]",ylabel="Quota sul riferimento dichiarato [m]",
       title=f"{'DEMO · ' if DEMO else ''}{identifier}: beta = {info.slope:.4f}, R² = {info.r_squared:.3f}")
ax.legend(fontsize=8); ax.grid(alpha=.2); fig.tight_layout()
fig.savefig(WORK/"profilo_selezionato.png",dpi=170)
PLOT_ANALYSIS_SIGNATURE=ANALYSIS_SIGNATURE
plt.show()

Controlla soprattutto che il primo punto sia dalla parte del mare e che i punti arancioni rappresentino la battigia. L’esclusione dietro il primo superamento di `Z_MAX` riprende il notebook didattico; non costituisce una simulazione della connessione idraulica o dell’inondazione.

In [ ]:
#@title Scegli i transetti e le pendenze da esportare
TRANSETTI_ESCLUSI = '' #@param {type:"string"}
MODALITA_PENDENZE = 'Valori dei transetti' #@param ["Valori dei transetti", "Intervallo minimo-massimo"]
NUM_PENDENZE = 5 #@param {type:"integer"}
excluded={v.strip() for v in TRANSETTI_ESCLUSI.split(",") if v.strip()}
unknown=excluded-set(riepilogo.transect_id)
if unknown:
    raise ValueError(f"ID da escludere non riconosciuti: {unknown}")
scelti=riepilogo[~riepilogo.transect_id.isin(excluded)].copy()
if scelti.empty or (scelti.status!="valid").any():
    raise ValueError("Correggi i transetti non validi o escludili esplicitamente prima di esportare.")
if MODALITA_PENDENZE=="Valori dei transetti":
    PENDENZE=sorted(set(float(value) for value in scelti.slope))
else:
    if NUM_PENDENZE<2:
        raise ValueError("Usa almeno due campioni per un intervallo.")
    PENDENZE=sorted(set(float(value) for value in np.linspace(scelti.slope.min(),scelti.slope.max(),NUM_PENDENZE)))
SELECTION_ANALYSIS_SIGNATURE=ANALYSIS_SIGNATURE
print("Pendenze da usare nel portale:",PENDENZE)
print("Gli eventuali valori identici sono deduplicati; gli scenari del portale non pesano i transetti per lunghezza di costa.")

## 5. Stagioni, strutture e riferimento delle quote
Le quote seguenti sono **esempi da definire per il sito**, come nella prima versione del portale. Indica inizio/fine della stagione in formato `MM-GG`; lascia entrambi vuoti se ancora sconosciuti. La correzione del livello marino raccorda il riferimento del modello a quello locale; non va usata per correggere automaticamente il DEM.

In [ ]:
#@title Configurazione delle soglie
INIZIO_ESTATE = '05-01' #@param {type:"string"}
FINE_ESTATE = '09-30' #@param {type:"string"}
QUOTA_OMBRELLONI_M = 0.7 #@param {type:"number"}
QUOTA_STRUTTURE_STAGIONALI_M = 0.9 #@param {type:"number"}
QUOTA_CHIOSCHI_M = 1.1 #@param {type:"number"}
QUOTA_PASSEGGIATA_M = 1.4 #@param {type:"number"}
RUGOSITA_M = 0.001 #@param {type:"number"}
OFFSET_LIVELLO_M = 0.0 #@param {type:"number"}
NOTE_SITO = 'Parametri di studio; quote e raccordo verticale da validare.' #@param {type:"string"}
import datetime as dt
if bool(INIZIO_ESTATE)!=bool(FINE_ESTATE):
    raise ValueError("Indica entrambe le date estive oppure nessuna.")
for value in (INIZIO_ESTATE,FINE_ESTATE):
    if value: dt.datetime.strptime("2000-"+value,"%Y-%m-%d")
heights=[QUOTA_OMBRELLONI_M,QUOTA_STRUTTURE_STAGIONALI_M,QUOTA_CHIOSCHI_M,QUOTA_PASSEGGIATA_M]
if not np.isfinite(heights).all() or heights!=sorted(set(heights)):
    raise ValueError("Le quattro quote devono essere finite e strettamente crescenti.")
if not np.isfinite([RUGOSITA_M,OFFSET_LIVELLO_M]).all() or RUGOSITA_M<0:
    raise ValueError("Rugosità o correzione del livello non valide.")
thresholds=[]
for identifier,height,label,season in zip(
    ["prima_fila_ombrelloni","strutture_turistiche_stagionali","strutture_turistiche_fisse","infrastrutture_fisse"],heights,
    ["Prima fila di ombrelloni","Strutture turistiche stagionali","Chioschi e strutture fisse","Passeggiata a mare"],
    ["summer","summer","all_year","all_year"]):
    thresholds.append(dict(id=identifier,elevation_m=float(height),short_label=label,description="Possibile interessamento: "+label.lower(),season=season))
display(pd.DataFrame(thresholds)[["short_label","elevation_m","season"]])

### 5A. Settori di spiaggia — facoltativi
Lascia `SETTORI_SPIAGGIA = {}` per rappresentare la spiaggia intera. Altrimenti assegna tutti i transetti inclusi a un solo settore ciascuno. Le pendenze sono ricavate dai transetti con il metodo già usato. Onde, ERA5, marea e surge restano condivisi; soglie, esposizione e stagione si ereditano dal sito, salvo diversa indicazione.

I settori di spiaggia sono distinti dai settori di provenienza delle onde. I confini non sono dedotti automaticamente dagli ID dei transetti.

In [ ]:
import copy
from forecast_service.sectors import resolve_sectors

def soglie_settore(quote):
    if len(quote) != len(thresholds):
        raise ValueError("Indicare una quota per ciascuna soglia del sito.")
    result = copy.deepcopy(thresholds)
    for threshold, value in zip(result, quote):
        threshold["elevation_m"] = float(value)
    return result

# Adatta ai tuoi transetti reali; tutti gli ID inclusi devono essere assegnati.
# SETTORI_SPIAGGIA = {
#   "ponente": {"name": "Spiaggia di Ponente", "transect_ids": ["1", "2"]},
#   "levante": {"name": "Spiaggia di Levante", "transect_ids": ["3", "4"],
#               "thresholds": soglie_settore([0.65, 0.85, 1.05, 1.35]),
#               "wave_exposure": {"sectors_deg": [[120, 220]], "note": "Da rilievo"}}
# }
SETTORI_SPIAGGIA = {}
preview_location = dict(SITE_INPUT, slopes=PENDENZE, roughness_m=float(RUGOSITA_M),
    transects=json.loads(scelti.to_json(orient="records")),
    wave_exposure=parse_exposure(SETTORI_PROVENIENZA, NOTE_ESPOSIZIONE), sectors=SETTORI_SPIAGGIA)
settori_risolti = resolve_sectors(preview_location, thresholds)
if settori_risolti:
    display(pd.DataFrame([{"settore":key,"nome":value["name"],"transetti":value["transect_ids"],"pendenze":value["slopes"]} for key,value in settori_risolti.items()]))
else:
    print("Spiaggia unica, senza suddivisioni.")

### 5B. DEM, ortofotomosaico e crediti — facoltativi
Lascia `CARTOGRAFIE = []` per non produrre mappe. Ogni voce può riferirsi alla spiaggia intera (`unit_id="whole"`) oppure a uno dei settori.

Il DEM deve essere un GeoTIFF locale ritagliato, con quote in **metri**, e includere una fascia attorno all’area di interesse per verificare le connessioni. L’ortofotomosaico è opzionale, GeoTIFF georeferenziato a 8 bit (grigio/RGB/RGBA). Si esportano soltanto una griglia e un’immagine leggere, normalmente fino a 768 pixel per asse; gli originali restano fuori dalla repository.

Indica un poligono GeoJSON e punti di partenza verso mare in WGS84 **[longitudine, latitudine]**. La connessione si calcola sull’intero DEM; il poligono ritaglia solo il risultato. I punti devono cadere su celle valide sufficientemente basse: NoData non è acqua e non viene riempito artificialmente.

Compila **autore/ente del rilievo, data, licenza e riferimento verticale**, distinguendo DEM e ortofoto se hanno autori diversi. Conferma `vertical_alignment_confirmed=True` solo dopo aver raccordato lo zero delle quote al TWL. `dem_offset_m` si somma al DEM una sola volta, indipendentemente dalla correzione dei profili. La griglia ridotta può perdere muri e varchi stretti: controlla l’anteprima e, se necessario, riduci l’area o usa fino a 1024 pixel.

Le estensioni usano le quote orarie P75 e P95 più alte nelle 96 ore: non sono probabilità né una simulazione dinamica di allagamento.

In [ ]:
#@title Carica i raster dal computer (facoltativo)
CARICA_RASTER_DAL_COMPUTER = False #@param {type:"boolean"}
if CARICA_RASTER_DAL_COMPUTER:
    from google.colab import files
    uploads = files.upload()
    for filename, content in uploads.items():
        destination = WORK / Path(filename).name
        destination.write_bytes(content)
        print(destination)

In [ ]:
from forecast_service.mapping_setup import prepare_mapping
# Aggiungi una voce per ogni unità con cartografia, usando geometrie reali.
# CARTOGRAFIE = [{
#   "unit_id": "ponente",  # oppure "whole"
#   "dem_path": str(WORK / "dem.tif"),
#   "orthophoto_path": str(WORK / "ortofotomosaico.tif"),  # oppure None
#   "display_polygon": {"type": "Polygon", "coordinates": [...]},
#   "seed_points": [[LONGITUDINE, LATITUDINE]],
#   "survey_credit": "DEM: autore / ente. Ortofoto: autore / ente.",
#   "survey_date": "AAAA-MM-GG", "license": "Licenza / attribuzione del rilievo",
#   "vertical_reference": "DEM raccordato allo zero del TWL",
#   "dem_offset_m": 0.0, "vertical_alignment_confirmed": True,
#   "max_dimension": 768
# }]
CARTOGRAFIE = []
MAPPING_CONFIGS = {}; CARTOGRAPHY_ASSETS = {}
for spec in CARTOGRAFIE:
    unit = spec.get("unit_id", "whole")
    if unit in MAPPING_CONFIGS or unit != "whole" and unit not in SETTORI_SPIAGGIA:
        raise ValueError("Unità cartografica duplicata o non definita.")
    metadata, assets = prepare_mapping(output=WORK/"cartography"/unit, slug=ID_SITO, **spec)
    MAPPING_CONFIGS[unit] = metadata; CARTOGRAPHY_ASSETS.update(assets)
    from forecast_service.inundation import load_grid
    grid, seeds, mask, bounds = load_grid(assets[f"cartography/{unit}/grid.npz"])
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.imshow(np.where(mask, grid, np.nan), cmap="terrain")
    yy, xx = np.nonzero(seeds); ax.scatter(xx, yy, color="red", label="Avvio verso mare")
    ax.set_title(f"{unit}: DEM e area visualizzata (non previsione)"); ax.legend(); display(fig); plt.close(fig)
CARTOGRAPHY_SIGNATURE = json.dumps(CARTOGRAFIE, sort_keys=True)
print("Mappe predisposte:", list(MAPPING_CONFIGS) or "nessuna")

## 6. ERA5 — scarica una volta o riusa il tuo CSV
Per dati reali servono un account Climate Data Store e l’accettazione personale delle condizioni di [ERA5 time-series](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=download). La ricerca automatica della cella marina usa anche [ERA5 grigliato](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels?tab=download).

Puoi salvare il token nei **Secrets di Colab**, con nome `CDS_KEY` e accesso a questo notebook, oppure inserirlo nel campo protetto quando richiesto. Non scriverlo nei campi del notebook. La cache contiene soltanto richieste e dati: sarà riutilizzata se la sessione riparte dalla stessa cartella.

Il servizio ERA5 restituisce **periodo medio**, non Tp: lo salviamo come `tm_s`; non lo usiamo come periodo di picco per il runup. Il confronto storico usa Hs. Il confronto con tutte le ore storiche è distinto dal fit Gumbel dei massimi annuali per settore descritto sotto.

### Tempi di ritorno delle onde per settore
La suddivisione statistica in settori di provenienza è distinta dall’esposizione fisica della spiaggia. Puoi scegliere l’ampiezza dei settori: nessuna direzione è privilegiata. I settori sono centrati su multipli dell’ampiezza scelta; per esempio, con 45°, quello centrato a 90° va da 67,5° incluso a 112,5° escluso. Le direzioni indicano **da dove arrivano** le onde.

Per ogni anno di calendario UTC completo si seleziona il massimo Hs **fra tutte le ore di ciascun settore**, non si suddivide il solo massimo annuale generale. Si usano soltanto anni con copertura oraria completa e direzione valida (la direzione può mancare nelle ore di calma con Hs=0). Se un settore non ha onde positive in qualche anno completo, il fit non viene mostrato: rimuovere quegli anni condizionerebbe la frequenza.

Il fit è una **Gumbel sui massimi annuali** (massima verosimiglianza), con banda bootstrap al 95% ricampionando gli anni. Questa prima versione richiede 30 anni completi; una serie lunga non dimostra da sola che Gumbel sia adeguata. Si assumono stazionarietà e indipendenza dei massimi annuali. Sul sito il picco Hs dei quattro giorni viene confrontato con il settore della sua direzione in quell’ora; si possono esaminare anche i picchi negli altri settori presenti.

Il tempo di ritorno riguarda Hs al largo per quel settore, non runup o allagamenti. La differenza fra ERA5 e Copernicus non è corretta e non rientra nella banda del fit. Non usare queste elaborazioni sperimentali per previsioni o scelte gestionali.

In [ ]:
#@title Impostazioni dei massimi annuali per settore
CALCOLA_ESTREMI_SETTORIALI = True #@param {type:"boolean"}
AMPIEZZA_SETTORI_ESTREMI = 45 #@param [30,45,60,90,120,180,360]
COL_DIREZIONE_ERA5 = 'direction_deg' #@param {type:"string"}
MIN_ANNI_ESTREMI = 30
REPLICHE_BOOTSTRAP = 400
print("Settori statistici:",AMPIEZZA_SETTORI_ESTREMI,"gradi; minimo",MIN_ANNI_ESTREMI,"anni completi.")

In [ ]:
#@title Sorgente e periodo storico
MODALITA_ERA5 = 'Scarica ERA5' #@param ["Scarica ERA5", "Importa CSV", "Salta per ora"]
ERA5_INIZIO = '1940-01-01' #@param {type:"date"}
ERA5_FINE = '2025-12-31' #@param {type:"date"}
CONSENSO_DOWNLOAD_ERA5 = True #@param {type:"boolean"}
CELLA_ERA5_NOTA = False #@param {type:"boolean"}
LAT_CELLA_ERA5 = 44.08 #@param {type:"number"}
LON_CELLA_ERA5 = 8.33 #@param {type:"number"}
FILE_CSV_ERA5 = '' #@param {type:"string"}
COL_TEMPO_ERA5 = 'time' #@param {type:"string"}
COL_HS_ERA5 = 'hs' #@param {type:"string"}
SCARTA_HS_MANCANTI = False #@param {type:"boolean"}
CLIMATE=None
CLIMATE_DATA_MODE="demo" if DEMO else "study"
climate_metadata={}
CACHE=BASE/"_cache_era5"
CACHE.mkdir(parents=True,exist_ok=True)
if DEMO:
    # Synthetic reference, clearly labelled; never presented as an ERA5 download.
    time=pd.date_range("2019-01-01", "2020-12-31 23:00",freq="h",tz="UTC")
    index=np.arange(len(time))
    hs=.25+1.1*(.5+.5*np.sin(index/179))**3+.3*(.5+.5*np.sin(index/37))
    era5_frame=pd.DataFrame({"time":time,"hs_m":hs,"direction_deg":(index*13)%360})
    history=WORK/"clima_dimostrativo.csv"; era5_frame.to_csv(history,index=False)
    CLIMATE=build_climate(history,"Esempio sintetico; non ERA5")
    climate_metadata={"mode":"demo","description":"Distribuzione sintetica per verificare il notebook."}
elif MODALITA_ERA5=="Scarica ERA5":
    if not CONSENSO_DOWNLOAD_ERA5:
        raise ValueError("Accetta le condizioni sul CDS e imposta CONSENSO_DOWNLOAD_ERA5=True per avviare la richiesta.")
    import os
    if not os.environ.get("CDS_KEY"):
        token=None
        try:
            from google.colab import userdata
            token=userdata.get("CDS_KEY")
        except Exception:
            pass
        if not token:
            from getpass import getpass
            token=getpass("Token CDS (input nascosto): ")
        if not token: raise ValueError("Token CDS non disponibile.")
        os.environ["CDS_KEY"]=token
        del token
    known_cell=(LON_CELLA_ERA5,LAT_CELLA_ERA5) if CELLA_ERA5_NOTA else None
    era5_frame,climate_metadata=download_series(SITE_INPUT,dt.date.fromisoformat(ERA5_INIZIO),dt.date.fromisoformat(ERA5_FINE),CACHE,cell=known_cell)
    history=WORK/"ERA5_hourly.csv"; era5_frame.to_csv(history,index=False)
    if climate_metadata["missing_hs_records"] and not SCARTA_HS_MANCANTI:
        raise ValueError("ERA5 ha ore mancanti: CSV conservato. Esamina i dati; puoi autorizzare lo scarto con SCARTA_HS_MANCANTI.")
    if SCARTA_HS_MANCANTI:
        climate_metadata["dropped_missing_rows"]=int(era5_frame.hs_m.isna().sum())
        era5_frame=era5_frame.dropna(subset=["hs_m"])
        history=WORK/"ERA5_hs_valide.csv"; era5_frame.to_csv(history,index=False)
    CLIMATE=build_climate(history,"ERA5")
elif MODALITA_ERA5=="Importa CSV":
    if not FILE_CSV_ERA5:
        from google.colab import files
        uploaded=files.upload()
        if len(uploaded)!=1: raise ValueError("Carica un solo CSV storico.")
        name,content=next(iter(uploaded.items()))
        history=WORK/Path(name).name; history.write_bytes(content)
    else:
        history=Path(FILE_CSV_ERA5)
    era5_frame=pd.read_csv(history)
    if SCARTA_HS_MANCANTI:
        climate_metadata["dropped_missing_rows"]=int(era5_frame[COL_HS_ERA5].isna().sum())
        era5_frame=era5_frame.dropna(subset=[COL_HS_ERA5])
        history=WORK/"ERA5_import_hs_valide.csv"; era5_frame.to_csv(history,index=False)
    CLIMATE=build_climate(history,"ERA5 (CSV dell’utente)",time_column=COL_TEMPO_ERA5,hs_column=COL_HS_ERA5)
else:
    print("Climatologia saltata: la località potrà essere esportata senza riferimento storico.")
if CLIMATE is not None:
    if CALCOLA_ESTREMI_SETTORIALI:
        imported=not DEMO and MODALITA_ERA5=="Importa CSV"
        CLIMATE["directional_extremes"]=build_directional_extremes(history,
            time_column=COL_TEMPO_ERA5 if imported else "time",
            hs_column=COL_HS_ERA5 if imported else "hs_m",
            direction_column=COL_DIREZIONE_ERA5 if imported else "direction_deg",
            sector_width_deg=int(AMPIEZZA_SETTORI_ESTREMI),min_years=MIN_ANNI_ESTREMI,
            bootstrap_replicates=REPLICHE_BOOTSTRAP)
        ext=CLIMATE["directional_extremes"]
        if ext["status"]=="available":
            display(pd.DataFrame([{"settore":f'{s["start_deg"]:g}° → {s["end_deg"]:g}°',
                "anni":s["fit"]["n_years"],"fit":s["fit"]["status"],"nota":s["fit"].get("reason","")}
                for s in ext["sectors"]]))
        else: print(ext["reason"])
    CLIMATE["download"]=climate_metadata
    if not DEMO: CLIMATE["doi"]="10.24381/1cf1ad76"
    (WORK/"climatology.json").write_text(json.dumps(CLIMATE,ensure_ascii=False,allow_nan=False,separators=(",",":")))
    display(pd.DataFrame([{k:CLIMATE[k] for k in ["source","start","end","n_records","p95_m","p99_m","max_hs_m","coverage_fraction"]}]))

In [ ]:
#@title Verifica la distribuzione e i massimi annuali
if CLIMATE is not None:
    fig,axes=plt.subplots(1,2,figsize=(12,4))
    axes[0].hist(CLIMATE["hs_sorted_m"],bins=50,color="#527d78",density=True)
    for key,label,color in [("p95_m","P95","#b39245"),("p99_m","P99","#b65f44")]:
        axes[0].axvline(CLIMATE[key],ls="--",color=color,label=f"{label}: {CLIMATE[key]:.2f} m")
    axes[0].set(xlabel="Hs [m]",ylabel="Densità",title=CLIMATE["source"])
    axes[0].legend()
    annual=pd.DataFrame.from_dict(CLIMATE["annual_maxima"],orient="index")
    axes[1].bar(annual.index,annual.hs_m,color="#527d78")
    axes[1].tick_params(axis="x",labelrotation=60)
    axes[1].set(xlabel="Anno",ylabel="Hs massima osservata [m]",title="Massimi annuali osservati, non valori di ritorno")
    fig.tight_layout(); fig.savefig(WORK/"climatologia.png",dpi=150); plt.show()
    annual.to_csv(WORK/"massimi_annuali.csv",index_label="year")
    display(annual)
    if "cell_lat" in climate_metadata:
        m=folium.Map(location=[LAT_ONDE,LON_ONDE],zoom_start=8)
        folium.Marker([LAT_SITO,LON_SITO],tooltip="Sito").add_to(m)
        folium.CircleMarker([LAT_ONDE,LON_ONDE],radius=6,color="green",tooltip="Punto onde richiesto").add_to(m)
        folium.CircleMarker([climate_metadata["cell_lat"],climate_metadata["cell_lon"]],radius=7,color="orange",tooltip="Cella marina ERA5 usata").add_to(m)
        display(m)
else:
    print("Nessuna climatologia selezionata.")

In [ ]:
#@title Verifica le curve Gumbel dei settori
ext=CLIMATE.get("directional_extremes") if CLIMATE else None
if ext and ext["status"]=="available":
    valid=[s for s in ext["sectors"] if s["fit"]["status"]=="available"]
    if valid:
        fig,axes=plt.subplots((len(valid)+1)//2,2,figsize=(12,4*((len(valid)+1)//2)),squeeze=False)
        for ax,sector in zip(axes.flat,valid):
            fit=sector["fit"]; periods=fit["return_period_years"]
            ax.fill_between(periods,fit["ci95_low_m"],fit["ci95_high_m"],alpha=.2,color="#527d78",label="Bootstrap 95%")
            ax.plot(periods,fit["return_level_m"],color="#194e4c",label="Gumbel")
            ax.scatter([r["return_period_years"] for r in fit["empirical"]],[r["hs_m"] for r in fit["empirical"]],s=12,label="Massimi annuali")
            ax.set(xscale="log",xlabel="Tempo di ritorno [anni]",ylabel="Hs al largo [m]",title=f'{sector["start_deg"]:g}° → {sector["end_deg"]:g}° · {fit["n_years"]} anni')
            ax.set_ylim(bottom=0); ax.legend(fontsize=8)
        for ax in list(axes.flat)[len(valid):]: ax.set_visible(False)
        fig.tight_layout(); fig.savefig(WORK/"estremi_settoriali.png",dpi=150); plt.show()
    else: print("Nessun settore soddisfa i requisiti per il fit. Esamina le note nella tabella.")
else: print("Estremi settoriali non configurati.")

### Zero del livello marino: riferimento storico fisso Copernicus
Questa sezione è separata da ERA5: usa la **SSH del modello fisico Copernicus**, che comprende la marea e le variazioni atmosferiche e oceanografiche. Si calcola una volta la media di un anno UTC completo nella stessa cella usata per le previsioni, e la si conserva per gli aggiornamenti futuri. Non si sottrae la media dei giorni di previsione.

Per Borghetto è incluso il riferimento già verificato del **2025**: 8.760 ore, media SSH circa −0,4032 m. Puoi usarlo senza scaricare nuovamente dati o rifare ERA5. Il controllo rifiuta di applicarlo se cambi il punto del livello marino. Per un altro sito scegli **Calcola da Copernicus** e un anno completo disponibile nell’archivio: il download richiede l’account Copernicus Marine, distinto da quello CDS/ERA5. Le credenziali si leggono dai Secrets Colab o da input protetto e non sono esportate.

Lo zero così ottenuto è la **media del modello nell’anno dichiarato**, non una calibrazione del livello medio mare misurato sulla spiaggia. Il raccordo con le quote delle strutture rimane provvisorio. Eventuali correzioni locali devono restare esplicite nel campo `OFFSET_LIVELLO_M`.

In [ ]:
#@title Riferimento storico del livello marino
MODALITA_RIFERIMENTO_SSH = 'Usa riferimento incluso' #@param ["Usa riferimento incluso", "Calcola da Copernicus", "Importa JSON"]
ANNO_RIFERIMENTO_SSH = 2025 #@param {type:"integer"}
CONSENSO_DOWNLOAD_SSH = False #@param {type:"boolean"}
FILE_RIFERIMENTO_SSH = '' #@param {type:"string"}
SEA_LEVEL_REFERENCE=None
reference_location=dict(SITE_INPUT)
if not LIVELLO_STESSO_PUNTO:
    reference_location["sealevel_point"]=dict(latitude=LAT_LIVELLO,longitude=LON_LIVELLO)
if DEMO:
    print("Modalità demo: il livello marino resta sintetico; nessun riferimento reale viene usato.")
elif MODALITA_RIFERIMENTO_SSH=="Usa riferimento incluso":
    reference_file=RUNTIME/"config"/"study_sites"/ID_SITO/f"sea_level_reference_{ANNO_RIFERIMENTO_SSH}.json"
    if not reference_file.exists():
        raise ValueError("Nessun riferimento incluso per questo sito/anno: scegli Calcola da Copernicus o Importa JSON.")
    SEA_LEVEL_REFERENCE=json.loads(reference_file.read_text())
elif MODALITA_RIFERIMENTO_SSH=="Importa JSON":
    if FILE_RIFERIMENTO_SSH:
        SEA_LEVEL_REFERENCE=json.loads(Path(FILE_RIFERIMENTO_SSH).read_text())
    else:
        from google.colab import files
        uploaded=files.upload()
        if len(uploaded)!=1: raise ValueError("Carica un solo riferimento JSON.")
        SEA_LEVEL_REFERENCE=json.loads(next(iter(uploaded.values())))
else:
    if not CONSENSO_DOWNLOAD_SSH:
        raise ValueError("Per scaricare la SSH storica imposta CONSENSO_DOWNLOAD_SSH=True.")
    from getpass import getpass
    def account_secret(name,prompt):
        import os
        value=os.environ.get(name)
        if not value:
            try:
                from google.colab import userdata
                value=userdata.get(name)
            except Exception:
                pass
        return value or getpass(prompt)
    username=account_secret("COPERNICUSMARINE_USERNAME","Username Copernicus Marine: ")
    password=account_secret("COPERNICUSMARINE_PASSWORD","Password Copernicus Marine: ")
    try:
        SEA_LEVEL_REFERENCE=download_reference(reference_location,int(ANNO_RIFERIMENTO_SSH),username,password,cache=BASE/"_cache_ssh")
    finally:
        del username,password
if SEA_LEVEL_REFERENCE is not None:
    validate_reference(SEA_LEVEL_REFERENCE,location=reference_location)
    if SEA_LEVEL_REFERENCE["year"]!=ANNO_RIFERIMENTO_SSH:
        raise ValueError("L’anno del riferimento JSON non corrisponde a quello selezionato.")
    (WORK/f"sea_level_reference_{ANNO_RIFERIMENTO_SSH}.json").write_text(json.dumps(SEA_LEVEL_REFERENCE,ensure_ascii=False,allow_nan=False,indent=2))
    display(pd.DataFrame([{"riferimento":f'Media SSH Copernicus {SEA_LEVEL_REFERENCE["year"]}',"media SSH (m)":SEA_LEVEL_REFERENCE["mean_ssh_m"],"ore":SEA_LEVEL_REFERENCE["n_records"],"cella":SEA_LEVEL_REFERENCE["selected_point"],"versione dataset":SEA_LEVEL_REFERENCE["dataset_version"]}]))
    print("Questo valore resterà fisso nei futuri bollettini. Il raccordo con le quote locali è ancora da verificare.")

## 7. Rivedi ed esporta il sito
Il pacchetto contiene configurazione, soglie, pendenze e relativa diagnostica, profili normalizzati e climatologia quando disponibile. **Conserva la cache o il CSV storico in Drive**: il pacchetto riporta il riferimento statistico, non tutti i NetCDF delle richieste.

Un pacchetto prodotto in modalità demo è marcato come tale e l’importatore lo rifiuta per impostazione predefinita. Nessun token entra nel pacchetto. Nessun file del portale viene modificato da Colab.

In [ ]:
#@title Riepilogo prima dell’esportazione
from forecast_service.time_utils import iso
import hashlib
current_signature=(Z_MIN,Z_MAX,ORIENTAMENTO,COL_ID,COL_DIST,COL_Z,COL_X,COL_Y,
    hashlib.sha256(pd.util.hash_pandas_object(profili,index=True).values.tobytes()).hexdigest())
if PROFILE_DATA_MODE!=("demo" if DEMO else "study") or (CLIMATE is not None and CLIMATE_DATA_MODE!=("demo" if DEMO else "study")):
    raise ValueError("Modalità cambiata: riesegui caricamento profili, pendenze e climatologia prima di esportare.")
if current_signature!=ANALYSIS_SIGNATURE or SELECTION_ANALYSIS_SIGNATURE!=ANALYSIS_SIGNATURE or PLOT_ANALYSIS_SIGNATURE!=ANALYSIS_SIGNATURE:
    raise ValueError("Parametri o profili cambiati: riesegui calcolo, grafico e selezione delle pendenze.")
normalized=profili.rename(columns={COL_ID:"TR_ID",COL_DIST:"distance",COL_Z:"SAMPLE_1",COL_X:"X",COL_Y:"Y"})[["TR_ID","distance","SAMPLE_1","X","Y"]]
normalized.to_csv(WORK/"profili_normalizzati.csv",index=False)
location=dict(SITE_INPUT,slopes=PENDENZE,roughness_m=float(RUGOSITA_M),parameters_status="provisional",
    wave_exposure=parse_exposure(SETTORI_PROVENIENZA,NOTE_ESPOSIZIONE),
    datum_offset_m=float(OFFSET_LIVELLO_M),vertical_reference_status="provisional",
    climatology_file=f"config/climatology/{ID_SITO}.json" if CLIMATE is not None else None,
    summer_season=dict(start_month_day=INIZIO_ESTATE or None,end_month_day=FINE_ESTATE or None,status="configured" if INIZIO_ESTATE else "to_be_defined"),
    transects=json.loads(scelti.to_json(orient="records")),notes=NOTE_SITO)
if SETTORI_SPIAGGIA:
    location["sectors"] = copy.deepcopy(SETTORI_SPIAGGIA)
resolve_sectors(location, thresholds)
if CARTOGRAPHY_SIGNATURE != json.dumps(CARTOGRAFIE, sort_keys=True):
    raise ValueError("Cartografia cambiata: riesegui la preparazione prima di esportare.")
for unit, metadata in MAPPING_CONFIGS.items():
    if unit == "whole":
        location["mapping"] = metadata
    elif unit in location.get("sectors", {}):
        location["sectors"][unit]["mapping"] = metadata
    else:
        raise ValueError("Settore rimosso: riesegui la preparazione della cartografia.")
if not LIVELLO_STESSO_PUNTO:
    location["sealevel_point"]=dict(latitude=LAT_LIVELLO,longitude=LON_LIVELLO)
if SEA_LEVEL_REFERENCE is not None:
    if SEA_LEVEL_REFERENCE["year"]!=ANNO_RIFERIMENTO_SSH:
        raise ValueError("Anno del riferimento cambiato: riesegui la sezione del livello marino prima di esportare.")
    location=apply_reference(location,SEA_LEVEL_REFERENCE)
payload=dict(schema_version=1,slug=ID_SITO,data_mode="demo" if DEMO else "study",location=location,thresholds=thresholds,
    provenance=dict(created_at=iso(dt.datetime.now(dt.timezone.utc)),slope_method=BEACHFACE_METHOD_VERSION,
        reference_notebook="Stockdon_runup_soluzione.ipynb, celle 24–25",profile_source=PROFILE_SOURCE,
        profile_sha256=hashlib.sha256((WORK/"profili_normalizzati.csv").read_bytes()).hexdigest(),
        profile_data_mode=PROFILE_DATA_MODE,climate_data_mode=CLIMATE_DATA_MODE if CLIMATE else None,
        sea_level_reference_year=SEA_LEVEL_REFERENCE["year"] if SEA_LEVEL_REFERENCE else None,
        profile_crs=crs.to_string(),vertical_reference=RIFERIMENTO_QUOTE,z_correction_m=CORREZIONE_QUOTE_DEM_M,
        excluded_transects=sorted(excluded),slope_sampling=MODALITA_PENDENZE,z_min_m=Z_MIN,z_max_m=Z_MAX))
display(pd.DataFrame([{"località":NOME_SITO,"modalità":payload["data_mode"],"transetti inclusi":len(scelti),"pendenze":PENDENZE,"ERA5 o riferimento":CLIMATE["source"] if CLIMATE else "Non configurato","esposizione":location["wave_exposure"]["sectors_deg"] if location["wave_exposure"] else "Da definire","stagione estiva":f"{INIZIO_ESTATE} / {FINE_ESTATE}" if INIZIO_ESTATE else "Da definire"}]))
print("Le soglie e il raccordo delle quote restano da validare, anche con profili misurati.")

In [ ]:
#@title Crea e scarica il pacchetto
ESPORTA_PACCHETTO = True #@param {type:"boolean"}
if ESPORTA_PACCHETTO:
    attachments={name:WORK/name for name in ["pendenze_transetti.csv","profili_normalizzati.csv","profilo_selezionato.png"]}
    for name in ["climatologia.png","massimi_annuali.csv","estremi_settoriali.png"]:
        if CLIMATE is not None and (WORK/name).exists(): attachments[name]=WORK/name
    output=WORK/f"{ID_SITO}_{'DEMO' if DEMO else 'studio'}.zip"
    write_site_package(output,payload,climatology=CLIMATE,attachments=attachments,cartography=CARTOGRAPHY_ASSETS)
    print("Pacchetto creato:",output)
    try:
        from google.colab import files
        files.download(str(output))
    except ImportError:
        display(FileLink(str(output)))

## 8. Importa il pacchetto nella repository
Dopo il download, dal terminale nella cartella `Runup_GUI`, per aggiornare una località già presente:

```sh
python scripts/import_site_package.py /percorso/borghetto-santo-spirito_studio.zip --replace
```

Per una località nuova ometti `--replace`. Il controllo della configurazione viene eseguito prima di scrivere i file. Per una prova locale con un pacchetto demo serve anche `--allow-demo`; non usarlo per pubblicare parametri di esempio come dati reali.

Dopo l’importazione rigenera il bollettino e verifica il risultato nel portale. Il download ERA5 rimane un’attività di setup: gli aggiornamenti giornalieri leggono il riferimento già salvato.

### Riferimenti
- Metodo della pendenza: notebook didattico fornito dall’utente **Stockdon_runup_soluzione.ipynb**, celle 24–25.
- Stockdon, H. F., Holman, R. A., Howd, P. A., & Sallenger, A. H. (2006). *Empirical parameterization of setup, swash, and runup*. Coastal Engineering, 53(7), 573–588. https://doi.org/10.1016/j.coastaleng.2005.12.005
- ERA5 hourly time-series: https://doi.org/10.24381/1cf1ad76. Il downloader adatta lo script fornito dall’utente, copyright 2026 Alessio Rovere, MIT; la licenza è inclusa nella cella tecnica.
- [Colab: caricamento, salvataggio e condivisione dei notebook](https://research.google.com/colaboratory/faq.html).